# Fusion strategies for PROTAC degradation targets

Six (plus one optional) ways of fusing molecular and biological-context blocks, compared on
**Dmax**, **pDC50** and **activity** with 5 repeats x 5 folds of scaffold-grouped,
stratified cross-validation, and statistically compared *within task*.

| section | content |
|---|---|
| 1 | Setup: seeds, versions, flags, cache location |
| 2 | Data: `load_data()` stub, real loader with cached context, synthetic data |
| 3 | Scaffold groups and cached CV splits |
| 4 | Split diagnostics and leakage test |
| 5 | Method definitions (preprocessing, M1-M7) and the experiment harness |
| 6 | Smoke test on synthetic data (tiny fold count) |
| 7-8 | Full run (resumable) and statistics |
| 9 | Conclusions |

**Flags** (environment variables override the defaults so the notebook can run headless):
`FUSION_FAST_MODE=1` (1 repeat x 3 folds), `FUSION_RUN_M7=0` (skip XGBoost),
`FUSION_TASKS=dmax,pdc50,activity`.

## 1. Setup

In [ ]:
import os, sys, json, time, pickle, hashlib, warnings, platform, traceback, itertools, logging
from pathlib import Path

# macOS: torch and xgboost each bundle their own libomp; using both multi-threaded in one process segfaults (in either
# import order). Single-threaded OpenMP removes the clash; parallelism comes from worker processes (FUSION_N_JOBS).
# This must be set before numpy/torch/xgboost are imported, i.e. run this cell first in a fresh kernel.
if sys.platform == "darwin":
    os.environ.setdefault("OMP_NUM_THREADS", "1")
import xgboost as xgb
import numpy as np
import pandas as pd
import scipy
import scipy.stats as st
import sklearn
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from dotenv import load_dotenv, find_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*does not have valid feature names.*")
logging.getLogger("transformers").setLevel(logging.ERROR)

# ------------------------------- run configuration --------------------------------
def _env_flag(name, default):
    return os.environ.get(name, str(int(default))).strip().lower() in {"1", "true", "yes"}

FAST_MODE = _env_flag("FUSION_FAST_MODE", False)   # 1 repeat x 3 folds, for debugging
RUN_M7 = _env_flag("FUSION_RUN_M7", True)          # optional XGBoost reference
USE_SYNTHETIC = _env_flag("FUSION_USE_SYNTHETIC", False)  # force synthetic data in load_data()
ALL_TASKS = ["dmax", "pdc50", "activity"]
TASKS = [t for t in os.environ.get("FUSION_TASKS", ",".join(ALL_TASKS)).split(",") if t]

N_REPEATS, N_FOLDS = (1, 3) if FAST_MODE else (5, 5)   # folds actually *run*
N_SPLITS = 5                                           # K of the K-fold generator (always 5)
REPEAT_SEEDS = [1000 + r for r in range(5)]            # random_state of StratifiedGroupKFold
GLOBAL_SEED = 20240601
N_STRAT_BINS = 5                                       # target quantile bins for stratification
INNER_FOLDS = 3                                        # GroupKFold(3) inner CV for tuning
MAX_PCA = 64                                           # cap on k per block
GP_N_ITER, GP_HYPER_POINTS = 60, 1200                   # GP: Adam steps per restart, rows used for hyper-parameter fitting
NN_MAX_EPOCHS, NN_PATIENCE = 200, 15                    # M5/M6 training budget and early-stopping patience
N_JOBS = int(os.environ.get("FUSION_N_JOBS", "1"))      # parallel worker processes for the harness

# Thresholds (Dmax in [0, 1]; pDC50 = -log10(DC50 [M]))
DMAX_THR = 0.80
PDC50_THR = 6.0
AUC_THRESHOLD = {"dmax": DMAX_THR, "pdc50": PDC50_THR}   # ROC-AUC of the regression tasks: label = target > threshold
PRIMARY_METRIC = {"dmax": "r2", "pdc50": "r2", "activity": "roc_auc"}   # metric the statistics are run on

def set_all_seeds(seed=GLOBAL_SEED):
    """Seed python, numpy and torch (if installed)."""
    import random
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
    except ImportError:
        pass

set_all_seeds()
try:
    from IPython.display import display, Markdown
except ImportError:                       # plain-python fallback (tests / scripts)
    display = lambda obj: print(obj.to_string() if hasattr(obj, "to_string") else obj)
    Markdown = lambda text: text
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.precision", 3)

# ------------------------------- paths and cache ----------------------------------
load_dotenv(find_dotenv(usecwd=True))  # reads TACKAI_CACHE from a .env file (cwd or any parent) if present
if "TACKAI_CACHE" not in os.environ:
    print("TACKAI_CACHE is not set (no .env found): falling back to ~/.cache/tackai, tackai's default cache directory")
CACHE_DIR = Path(os.environ.get("TACKAI_CACHE", Path.home() / ".cache" / "tackai")).expanduser()
CACHE_DIR.mkdir(parents=True, exist_ok=True)
REPO_ROOT = Path(os.environ["TACK_REPO_ROOT"]) if "TACK_REPO_ROOT" in os.environ else next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DEV_FILES = [REPO_ROOT / "data/yaochen/AutoTPDplus-PROTAC-training.csv",
             REPO_ROOT / "data/yaochen/TACKv2.csv"]
POI_ESM_MODEL = "facebook/esm2_t30_150M_UR50D"   # 640-d; no ESM-2 checkpoint has 768 dims
CELL_ST_MODEL = "sentence-transformers/all-mpnet-base-v1"  # 768-d, via tackai.CellEmbedding
FP_RADIUS, FP_SIZE = 16, 1024

RESULTS_DIR = Path.cwd() / ("fusion_results_fast" if FAST_MODE else "fusion_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------- library versions ---------------------------------
def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return "n/a"

VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "scipy": scipy.__version__,
            "pandas": pd.__version__, "sklearn": sklearn.__version__, "matplotlib": matplotlib.__version__,
            "seaborn": sns.__version__, **{m: _ver(m) for m in ["torch", "gpytorch", "pingouin", "statsmodels",
                                                                  "rdkit", "xgboost", "transformers"]}}
(RESULTS_DIR / "versions.json").write_text(json.dumps({**VERSIONS, "seed": GLOBAL_SEED, "repeat_seeds": REPEAT_SEEDS}, indent=1))
print(json.dumps(VERSIONS, indent=1))
print(f"FAST_MODE={FAST_MODE}  RUN_M7={RUN_M7}  TASKS={TASKS}  repeats x folds = {N_REPEATS} x {N_FOLDS}")
print(f"TACKAI_CACHE -> {CACHE_DIR}\nRESULTS_DIR  -> {RESULTS_DIR}")

## 2. Data

### 2.0 `load_data()`: plug in your own data here
**`load_data()` (next cell) is the single entry point to replace with your own loader.** It must return a dict with the
row-aligned blocks `fingerprint, descriptors, e3, cell, poi, assay, assay_time` (2-D float arrays; `assay_time` is raw
hours with NaN where missing, NaNs in other blocks are mean-imputed per fold), `smiles` (1-D, used only for
grouping) and `y` (dict `dmax` in [0, 1], `pdc50`, `activity` in {0, 1}; NaN where a target is undefined for a row;
`make_targets(dmax_pct, dc50_nM)` builds it from the raw values). The default loader is defined further down
(section 2.3); `make_synthetic_data` (section 2.4) produces the same schema for smoke tests.

In [ ]:
# =====================================================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  USER STUB: PLUG IN YOUR OWN LOADER  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# =====================================================================================================
def load_data():
    """Return the data dict (schema in the markdown cell above).

    Replace the body with your own loader. The default reads the two development CSVs through the cached,
    target-independent context (``load_tack_dev_data``) or, with ``FUSION_USE_SYNTHETIC=1``, builds synthetic data.
    """
    if USE_SYNTHETIC:
        return make_synthetic_data(n=2500)
    return load_tack_dev_data()
# =====================================================================================================


### 2.1 Targets
* **Dmax**: reported in percent, scaled to [0, 1]. Scored with R² (and ROC-AUC at `Dmax > 0.8`).
* **pDC50** = -log10(DC50 [nM] x 1e-9) = 9 - log10(DC50 [nM]). Scored with R² (and ROC-AUC at `pDC50 > 6`).
  Note that pDC50 = 6 corresponds to DC50 = **1000 nM** (1 uM); the "DC50 = 100 nM" gloss in the task
  description would be pDC50 = 7. The numeric threshold 6 is used throughout. `DC50 <= 0` has no pDC50 -> NaN.
* **activity** = 1 if Dmax > 80 % **and** pDC50 > 6, else 0. If one of the two is missing the entry is
  inactive when the known one is strictly below its threshold (Dmax < 80 %, pDC50 < 6); otherwise it is undefined
  (NaN) and excluded from this task. Censoring operators (`>`, `<`, `~`) in the raw data are ignored.

Each task uses the rows where its own target is defined; the *context* (features) is shared by all tasks.

In [ ]:
def make_targets(dmax_pct, dc50_nM):
    """Build the three task targets from raw Dmax [%] and DC50 [nM].

    Args:
        dmax_pct: array-like of Dmax in percent (NaN if unknown).
        dc50_nM: array-like of DC50 in nM (NaN if unknown).

    Returns:
        dict with float arrays ``dmax`` ([0, 1] scale), ``pdc50`` and ``activity`` ({0, 1}); NaN = undefined.
    """
    dmax_pct = np.asarray(dmax_pct, dtype=float)
    dc50_nM = np.asarray(dc50_nM, dtype=float)
    dmax = dmax_pct / 100.0
    pdc50 = np.full_like(dc50_nM, np.nan)
    ok = dc50_nM > 0                                  # NaN compares False
    pdc50[ok] = 9.0 - np.log10(dc50_nM[ok])           # == -log10(DC50[nM] * 1e-9)

    d_known, p_known = ~np.isnan(dmax), ~np.isnan(pdc50)
    activity = np.full(len(dmax), np.nan)
    both = d_known & p_known
    activity[both] = ((dmax[both] > DMAX_THR) & (pdc50[both] > PDC50_THR)).astype(float)
    only_d, only_p = d_known & ~p_known, p_known & ~d_known
    activity[only_d & (dmax < DMAX_THR)] = 0.0
    activity[only_p & (pdc50 < PDC50_THR)] = 0.0
    return {"dmax": dmax, "pdc50": pdc50, "activity": activity}

In [ ]:
# TEST (written first): target construction
def _test_make_targets():
    nan = np.nan
    # columns: Dmax [%], DC50 [nM]
    cases = np.array([
        [90.0, 10.0],      # active: Dmax>80, pDC50=8>6
        [90.0, 1000.0],    # pDC50 == 6 exactly -> not > 6 -> inactive
        [90.0, 5000.0],    # weak potency -> inactive
        [50.0, 10.0],      # weak Dmax -> inactive
        [80.0, 10.0],      # Dmax == 80 -> not > 80 -> inactive (both known)
        [nan, 5000.0],     # Dmax missing, pDC50 = 5.3 < 6 -> inactive
        [nan, 10.0],       # Dmax missing, pDC50 passes -> undefined
        [30.0, nan],       # DC50 missing, Dmax < 80 -> inactive
        [95.0, nan],       # DC50 missing, Dmax passes -> undefined
        [80.0, nan],       # DC50 missing, Dmax == 80: not < 80 -> undefined
        [nan, nan],        # nothing known -> undefined everywhere
        [90.0, 0.0],       # DC50 <= 0 -> pDC50 undefined, Dmax passes -> undefined
    ])
    t = make_targets(cases[:, 0], cases[:, 1])
    assert set(t) == {"dmax", "pdc50", "activity"}
    np.testing.assert_allclose(t["dmax"][:3], [0.9, 0.9, 0.9])
    assert np.isnan(t["dmax"][5])
    np.testing.assert_allclose(t["pdc50"][[0, 1, 2]], [8.0, 6.0, -np.log10(5000e-9)])
    assert t["pdc50"][1] == 6.0, "pDC50 at DC50=1000 nM must be exactly 6 (threshold is strict)"
    assert np.isnan(t["pdc50"][11]), "DC50 <= 0 has no pDC50"
    act = t["activity"]
    expected = [1, 0, 0, 0, 0, 0, np.nan, 0, np.nan, np.nan, np.nan, np.nan]
    np.testing.assert_array_equal(np.nan_to_num(act, nan=-1), np.nan_to_num(np.array(expected, float), nan=-1))
    print("make_targets: OK")

_test_make_targets()

### 2.2 Small blocks: E3 ligase and assay one-hots
Vocabularies are fixed by rule (not learned from data), so one-hot encoding needs no fitting and cannot leak.
`assay_time` is stored raw (hours, NaN when unreported); its imputation/standardisation is fitted per fold.

In [ ]:
import re

E3_VOCAB = ["CRBN", "VHL", "IAP", "MDM2", "DCAF", "KEAP1", "FEM1B", "OTHER", "UNKNOWN"]
_E3_RULES = [("CRBN", r"^crbn$"), ("VHL", r"^vhl$"), ("IAP", r"iap|birc"), ("MDM2", r"^mdm2$"),
             ("DCAF", r"^dcaf"), ("KEAP1", r"^keap1$"), ("FEM1B", r"^fem1b$")]

ASSAY_VOCAB = ["western_blot", "in_cell_western", "capillary_immunoassay", "hibit", "nanoluc_reporter", "htrf",
               "elisa", "msd", "flow_cytometry", "high_content", "imaging", "mass_spec", "other", "unknown"]
# ordered: the first matching rule wins
_ASSAY_RULES = [("in_cell_western", r"in[- ]?cell western|incell western|in-cell elisa|cytoblot"),
                ("capillary_immunoassay", r"capillary|simple western|jess|\bwes\b"),
                ("hibit", r"hibit"),
                ("nanoluc_reporter", r"nanoluc|nano-glo|luciferase|luminescen"),
                ("htrf", r"htrf"),
                ("flow_cytometry", r"flow cytometry|facs"),
                ("high_content", r"high[- ]content|operetta|incell hunter"),
                ("western_blot", r"western|immunoblot|dot blot|licor"),
                ("elisa", r"elisa"),
                ("msd", r"\bmsd\b"),
                ("imaging", r"immunofluorescence|imaging|microscop"),
                ("mass_spec", r"mass spec|proteomic|lc-ms")]

def _missing(v):
    return v is None or (isinstance(v, float) and np.isnan(v)) or (isinstance(v, str) and not v.strip())

def _one_hot(labels, vocab):
    out = np.zeros((len(labels), len(vocab)), dtype=np.float32)
    idx = {v: i for i, v in enumerate(vocab)}
    for r, lab in enumerate(labels):
        out[r, idx[lab]] = 1.0
    return out

def canonical_e3(raw):
    """Map a raw recruiter/E3 name to a category of ``E3_VOCAB``."""
    if _missing(raw):
        return "UNKNOWN"
    s = str(raw).strip().lower()
    return next((name for name, pat in _E3_RULES if re.search(pat, s)), "OTHER")

def canonical_assay(raw):
    """Map free-text assay descriptions to a category of ``ASSAY_VOCAB`` (first matching rule wins)."""
    if _missing(raw):
        return "unknown"
    s = str(raw).strip().lower()
    return next((name for name, pat in _ASSAY_RULES if re.search(pat, s)), "other")

def encode_e3(raw):
    return _one_hot([canonical_e3(r) for r in raw], E3_VOCAB)

def encode_assay(raw):
    return _one_hot([canonical_assay(r) for r in raw], ASSAY_VOCAB)

In [ ]:
# TEST (written first): fixed-vocabulary one-hot encoders for the small blocks
def _test_onehots():
    # E3: aliases collapse, rare ligases -> OTHER, missing -> UNKNOWN
    e3 = encode_e3(["CRBN", "VHL", "XIAP", "cIAP1", "IAP", "MDM2", "DCAF16", "DCAF1", "Keap1", "FEM1B",
                    "UBR1", "HRD1", None, np.nan])
    assert e3.shape == (14, len(E3_VOCAB)) and (e3.sum(1) == 1).all()
    idx = e3.argmax(1)
    names = [E3_VOCAB[i] for i in idx]
    assert names[:2] == ["CRBN", "VHL"]
    assert names[2:5] == ["IAP"] * 3 and names[6:8] == ["DCAF"] * 2
    assert names[10:12] == ["OTHER"] * 2 and names[12:] == ["UNKNOWN"] * 2
    # assay: raw free text -> canonical categories
    cases = {
        "Western blot": "western_blot", "Western Blot/CellTiterGlo": "western_blot", "immunoblotting (Western blot)": "western_blot",
        "Licor-quantification of Western blots": "western_blot", "CCK8/Western Blot": "western_blot",
        "In-Cell Western": "in_cell_western", "In-Cell Western/Western Blot": "in_cell_western", "in-cell ELISA": "in_cell_western",
        "Automated Capillary Electrophoresis Immunoassay": "capillary_immunoassay", "Simple Western immunoassay (capillary-based)": "capillary_immunoassay",
        "HiBiT": "hibit", "HiBiT luciferase degradation assay (Nano-Glo HiBiT Lytic Detection)": "hibit", "Nano-Glo HiBiT": "hibit",
        "NanoLuc degradation kinetics": "nanoluc_reporter", "NanoLuc reporter assay (BTK-nLuc), endpoint": "nanoluc_reporter",
        "HTRF": "htrf", "HTRF LTC (L858R / T790M / C797S triple mutation)": "htrf",
        "ELISA": "elisa", "AR ELISA": "elisa", "MSD assay": "msd", "MSD (+elacridar pretreatment)": "msd",
        "Flow Cytometry": "flow_cytometry", "EGFP/mCherry reporter flow cytometry": "flow_cytometry", "Flow Cytometry, FACS": "flow_cytometry",
        "High-Content Analysis": "high_content", "Imaging-based GFP/mCherry ratio (high-content imaging, Operetta)": "high_content",
        "Immunofluorescence": "imaging", "Fluorescence Imaging Assay": "imaging", "Live-cell fluorescence imaging of GFP-BRD4": "imaging",
        "Standard MTS - based colorimetric assay": "other", None: "unknown", np.nan: "unknown", "": "unknown",
    }
    enc = encode_assay(list(cases))
    assert enc.shape == (len(cases), len(ASSAY_VOCAB)) and (enc.sum(1) == 1).all()
    got = [ASSAY_VOCAB[i] for i in enc.argmax(1)]
    bad = {k: (g, w) for k, g, w in zip(cases, got, cases.values()) if g != w}
    assert not bad, f"assay mapping mismatches (got, want): {bad}"
    assert 12 <= len(ASSAY_VOCAB) <= 16 and 8 <= len(E3_VOCAB) <= 12
    print(f"one-hot encoders: OK (E3 dims={len(E3_VOCAB)}, assay dims={len(ASSAY_VOCAB)})")

_test_onehots()

### 2.3 Development data and the cached, target-independent context
`load_tack_dev_data()` concatenates the two development CSVs (no de-duplication, as specified; rows without a
SMILES cannot be featurised and are dropped). All *context* blocks plus the molecular blocks are computed once
for **every** row, whatever its targets, and cached as individual `.npy` elements in
`TACKAI_CACHE/fusion_context/<key>/`. The key hashes the molecules and context columns only, never a label,
so every task and every future run reuses the same arrays. Entity-level embeddings are cached too
(ESM per unique sequence, sentence embedding per cell line, Morgan/descriptors per SMILES).

In [ ]:
BLOCK_ORDER = ["fingerprint", "descriptors", "e3", "cell", "poi", "assay", "assay_time"]
DENSE_BLOCKS = ["fingerprint", "descriptors", "cell", "poi"]   # scaled, optionally PCA-reduced
SMALL_BLOCKS = ["e3", "assay", "assay_time"]                   # never reduced, always passed through
CONTEXT_CACHE_VERSION = "v1"

def _first_valid(*series):
    out = series[0].copy()
    for s in series[1:]:
        out = out.where(out.notna() & (out.astype(str).str.strip() != ""), s)
    return out

def build_dev_table(files=None):
    """Read and harmonise the development CSVs into one row-per-record table (no de-duplication)."""
    files = list(files or DEV_FILES)
    parts = []
    for f in files:
        raw = pd.read_csv(f, low_memory=False)
        units = set(raw["DC50_units"].dropna().unique())
        assert units <= {"nM"}, f"{f.name}: unexpected DC50 units {units}"
        if "Cell_Line_STD_Accession" in raw:                       # AutoTPD+ layout
            cell_key = _first_valid(raw["Cell_Line_STD_Accession"], raw["Cell_Line_STD"], raw["Cell_Line"])
        else:                                                      # TACKv2 layout
            cell_key = _first_valid(raw["Cell_Line_ID"], raw["Cell_Line"])
        hours = raw[["DC50_h", "Dmax_h"]].astype(float)
        parts.append(pd.DataFrame({
            "source": f.stem, "smiles": raw["SMILES"], "e3_raw": _first_valid(raw["Recruiter"], raw["Recruiter_Gene"]),
            "poi_uniprot": raw["Degradation_Target_Uniprot"], "poi_seq": raw["Degradation_Target_Sequence"],
            "cell_key": cell_key, "assay_raw": raw["Assay"],
            "assay_time": hours.mean(axis=1, skipna=True),          # DC50_h and Dmax_h agree in ~98 % of rows
            "dmax_pct": raw["Dmax"].astype(float), "dc50_nM": raw["DC50"].astype(float)}))
    tab = pd.concat(parts, ignore_index=True)
    n0 = len(tab)
    tab = tab[tab["smiles"].notna() & (tab["smiles"].astype(str).str.strip() != "")].reset_index(drop=True)
    # fill a missing sequence from other rows of the same UniProt entry (most frequent sequence)
    have = tab.dropna(subset=["poi_uniprot", "poi_seq"])
    uni2seq = have.groupby("poi_uniprot")["poi_seq"].agg(lambda s: s.value_counts().index[0])
    miss = tab["poi_seq"].isna() & tab["poi_uniprot"].isin(uni2seq.index)
    tab.loc[miss, "poi_seq"] = tab.loc[miss, "poi_uniprot"].map(uni2seq)
    print(f"dev table: {n0} rows read, {n0 - len(tab)} dropped (no SMILES) -> {len(tab)}; "
          f"{tab['smiles'].nunique()} unique SMILES; {tab['poi_seq'].isna().sum()} rows without POI sequence; "
          f"{tab['cell_key'].isna().sum()} without cell line; {tab['assay_raw'].isna().mean():.0%} without assay; "
          f"{tab['assay_time'].isna().mean():.0%} without assay time")
    return tab

def context_key(tab):
    """Hash of everything the cached blocks depend on (molecules + context + featuriser config); never a label."""
    cols = ["smiles", "e3_raw", "poi_seq", "cell_key", "assay_raw", "assay_time"]
    h = hashlib.sha1(json.dumps([CONTEXT_CACHE_VERSION, POI_ESM_MODEL, CELL_ST_MODEL, FP_RADIUS, FP_SIZE,
                                 E3_VOCAB, ASSAY_VOCAB]).encode())
    h.update(tab[cols].astype(str).to_csv(index=False).encode())
    return h.hexdigest()[:12]

def context_dir(tab):
    return CACHE_DIR / "fusion_context" / f"dev_{context_key(tab)}"

In [ ]:
def embed_proteins_esm(seqs, model_name=POI_ESM_MODEL, max_len=1022, batch_size=8, cache_file=None):
    """Mean-pooled ESM-2 embedding per unique sequence, cached in ``TACKAI_CACHE`` (key = sha1 of the sequence).

    Sequences longer than ``max_len`` residues are cut into consecutive windows; the embedding is the
    residue-weighted mean over windows (BOS/EOS excluded).

    Args:
        seqs: iterable of amino-acid strings.
        model_name: Hugging Face ESM-2 checkpoint.
        max_len: residues per window (ESM-2 context is 1024 tokens incl. BOS/EOS).
        batch_size: windows per forward pass.
        cache_file: npz path of the entity cache (default derived from the model name).

    Returns:
        dict sequence -> float32 vector.
    """
    import torch
    from transformers import AutoTokenizer, AutoModel
    from tackai.data.embeddings.protein_embeddings import sanitize_sequence
    cache_file = Path(cache_file or CACHE_DIR / f"fusion_poi_{model_name.split('/')[-1]}_mean.npz")
    store = dict(np.load(cache_file)) if cache_file.exists() else {}
    seqs = list(dict.fromkeys(seqs))
    key = {s: hashlib.sha1(sanitize_sequence(s).encode()).hexdigest()[:20] for s in seqs}
    todo = [s for s in seqs if key[s] not in store]
    if todo:
        device = "mps" if torch.backends.mps.is_available() else "cpu"
        tok = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name).eval().to(device)
        windows = []   # (seq index, window string)
        for i, s in enumerate(todo):
            s = sanitize_sequence(s)
            windows += [(i, s[j:j + max_len]) for j in range(0, len(s), max_len)]
        windows.sort(key=lambda w: len(w[1]))
        acc = {i: (np.zeros(model.config.hidden_size, np.float64), 0) for i in range(len(todo))}
        for b in tqdm(range(0, len(windows), batch_size), desc=f"ESM {model_name.split('/')[-1]}", leave=False):
            batch = windows[b:b + batch_size]
            enc = tok([w for _, w in batch], return_tensors="pt", padding=True).to(device)
            with torch.no_grad():
                hid = model(**enc).last_hidden_state.float().cpu().numpy()
            mask = enc["attention_mask"].cpu().numpy().astype(bool)
            for (i, w), h, m in zip(batch, hid, mask):
                res = h[m][1:-1]                                    # drop BOS/EOS
                tot, n = acc[i]
                acc[i] = (tot + res.sum(0), n + len(res))
        for i, s in enumerate(todo):
            tot, n = acc[i]
            store[key[s]] = (tot / max(n, 1)).astype(np.float32)
        np.savez(cache_file, **store)
        del model
    return {s: store[key[s]] for s in seqs}

In [ ]:
def _mol_blocks(smiles):
    """Morgan bits and RDKit descriptors per unique SMILES through tackai's cached embedders."""
    from tackai.data.embeddings.mol_embeddings import MolEmbedding
    uniq = list(dict.fromkeys(smiles))
    fp = MolEmbedding(embeddings_type="fingerprint", radius=FP_RADIUS, fp_size=FP_SIZE, load_from_cache=True,
                      filename=f"morgan_fp_radius{FP_RADIUS}_size{FP_SIZE}.npz", cache_dir=str(CACHE_DIR))
    de = MolEmbedding(embeddings_type="rdkit_descriptors", load_from_cache=True,
                      filename="rdkit_descriptors.npz", cache_dir=str(CACHE_DIR))
    fp_map = fp.transform(uniq, update_cache=True)
    de_map = de.transform(uniq, update_cache=True)
    return (np.stack([fp_map[s] for s in smiles]).astype(np.float32),
            np.stack([de_map[s] for s in smiles]).astype(np.float32))

def _cell_block(cell_keys):
    """Sentence-transformer embedding of the Cellosaurus description (tackai CellEmbedding, cached)."""
    from tackai.data.embeddings.cell_embeddings import CellEmbedding
    keys = [None if _missing(k) else str(k) for k in cell_keys]
    ce = CellEmbedding(embeddings_type="sentence_transformer", pooling="sum", pretrained_model=CELL_ST_MODEL,
                       load_from_cache=True, filename="cell_embeddings_model=sentence-transformer_pooling=sum.npz",
                       cache_dir=str(CACHE_DIR))
    emb = ce.transform(list(dict.fromkeys(keys)), update_cache=True)
    return np.stack([emb[k] for k in keys]).astype(np.float32)

def _poi_block(seqs):
    uniq = [s for s in dict.fromkeys(seqs) if not _missing(s)]
    emb = embed_proteins_esm(uniq)
    dim = len(next(iter(emb.values())))
    return np.stack([emb[s] if not _missing(s) else np.full(dim, np.nan, np.float32) for s in seqs]).astype(np.float32)

def load_tack_dev_data(force=False):
    """Development data as the row-aligned block dict (see ``load_data``), with cached context elements."""
    tab = build_dev_table()
    d = context_dir(tab)
    if force or not all((d / f"{b}.npy").exists() for b in BLOCK_ORDER):
        t0 = time.time()
        d.mkdir(parents=True, exist_ok=True)
        fp, de = _mol_blocks(tab["smiles"].tolist())
        blocks = {"fingerprint": fp, "descriptors": de, "e3": encode_e3(tab["e3_raw"]),
                  "cell": _cell_block(tab["cell_key"]), "poi": _poi_block(tab["poi_seq"].tolist()),
                  "assay": encode_assay(tab["assay_raw"]),
                  "assay_time": tab[["assay_time"]].to_numpy(np.float32)}
        for b, arr in blocks.items():
            np.save(d / f"{b}.npy", arr)
        (d / "manifest.json").write_text(json.dumps({
            "key": context_key(tab), "n_rows": len(tab), "dims": {b: int(a.shape[1]) for b, a in blocks.items()},
            "poi_esm_model": POI_ESM_MODEL, "cell_model": CELL_ST_MODEL, "fp": [FP_RADIUS, FP_SIZE],
            "e3_vocab": E3_VOCAB, "assay_vocab": ASSAY_VOCAB, "version": CONTEXT_CACHE_VERSION}, indent=1))
        print(f"context computed and cached in {d} ({time.time() - t0:.0f}s)")
    else:
        print(f"context loaded from cache {d}")
    data = {b: np.load(d / f"{b}.npy") for b in BLOCK_ORDER}
    data["smiles"] = tab["smiles"].to_numpy(object)
    data["y"] = make_targets(tab["dmax_pct"], tab["dc50_nM"])
    data["meta"] = tab[["source", "poi_uniprot", "cell_key", "e3_raw", "assay_raw"]].reset_index(drop=True)
    return data

In [ ]:
# TEST (written first): real development data + target-independent cached context
def _test_dev_data():
    tab = build_dev_table()
    assert tab["smiles"].notna().all() and len(tab) > 5000
    assert set(tab["source"]) == {"AutoTPDplus-PROTAC-training", "TACKv2"}
    # The cache key must not depend on any label: same context -> same cache, whatever the targets.
    key = context_key(tab)
    t2 = tab.copy(); t2["dmax_pct"] = 1.0; t2["dc50_nM"] = 2.0
    assert context_key(t2) == key, "context cache key must be independent of the targets"
    t3 = tab.copy(); t3.loc[t3.index[0], "smiles"] = "CCO"
    assert context_key(t3) != key, "context cache key must change when the molecules change"

    data = load_tack_dev_data()
    n = len(data["smiles"])
    assert n == len(tab)
    for b in BLOCK_ORDER:
        assert data[b].ndim == 2 and data[b].shape[0] == n, b
    assert data["fingerprint"].shape[1] == FP_SIZE and set(np.unique(data["fingerprint"])) <= {0.0, 1.0}
    assert data["descriptors"].shape[1] > 150 and data["assay_time"].shape[1] == 1
    assert data["cell"].shape[1] == 768 and data["poi"].shape[1] >= 320
    assert (data["e3"].sum(1) == 1).all() and (data["assay"].sum(1) == 1).all()
    assert np.isfinite(data["cell"]).all() and np.isfinite(data["fingerprint"]).all()
    assert np.isnan(data["assay_time"]).any(), "assay_time must be stored raw (NaN kept) - imputation is per fold"
    # POI missing only where neither a sequence nor a UniProt-derived sequence exists (rare; mean-imputed per fold)
    assert np.isnan(data["poi"]).all(1).mean() < 0.02
    assert set(data["y"]) == {"dmax", "pdc50", "activity"} and all(len(v) == n for v in data["y"].values())
    # context cache on disk: one file per element, written once
    ctx_dir = context_dir(tab)
    for b in BLOCK_ORDER:
        assert (ctx_dir / f"{b}.npy").exists(), b
    t0 = time.time(); data2 = load_tack_dev_data(); dt = time.time() - t0
    assert dt < 60, f"second load should hit the cache (took {dt:.0f}s)"
    for b in BLOCK_ORDER:
        np.testing.assert_array_equal(np.nan_to_num(data[b], nan=-9), np.nan_to_num(data2[b], nan=-9))
    print(f"dev data: OK  n={n}  cache={ctx_dir}  (cached reload {dt:.1f}s)")

if USE_SYNTHETIC or not all(f.exists() for f in DEV_FILES):
    print("skipped: synthetic mode or development CSVs not found (" + ", ".join(str(f) for f in DEV_FILES) + ")")
else:
    _test_dev_data()

### 2.4 Synthetic data (smoke tests)
`make_synthetic_data(n)` mimics the schema with a **known** signal, so the pipeline (and the expected ranking of
the methods) can be checked before real data is attached:

`score = additive(mol, cell, poi, e3, assay) + 0.7 * mol_2 * poi_0 + 0.5 * cell_1 * poi_1 + noise`

The two product terms are genuine cross-block interactions (molecule x protein, cell line x protein); they are
uncorrelated with every block's own linear effect, so an additive model cannot see them. Molecules are real
SMILES built from ring/linker templates drawn from a skewed (Zipf-like) distribution, which gives scaffold groups of
very different sizes, plus a few acyclic molecules to exercise the failed-scaffold path. The block latent of a
molecule is mostly inherited from its scaffold, so scaffold-grouped CV is genuinely harder than random CV.

In [ ]:
# (ring SMILES, index after which a substituent may be attached: a ring carbon that is neither the entry nor the exit atom)
_SYN_RINGS = [("c1ccccc1", 3), ("c1ccncc1", 3), ("C1CCNCC1", 3), ("c1ccc2ccccc2c1", 3), ("C1CCCCC1", 3),
              ("c1cc[nH]c1", 3), ("C1CCOC1", 3), ("c1ncncc1", 4)]
_SYN_DECOR = ["", "(F)", "(Cl)", "(C)", "(OC)"]
_SYN_ACYCLIC = ["CCO", "CCN", "CCCC", "CC(C)O", "CCOC", "CCCN"]

def _syn_smiles(rng, template):
    a, k1, b, k2, c = template
    def dec(r):
        ring, slot = _SYN_RINGS[r]
        return ring[:slot] + _SYN_DECOR[rng.integers(len(_SYN_DECOR))] + ring[slot:]
    link = lambda k: "".join(rng.choice(["C", "C", "O", "N"]) if 0 < i < k - 1 else "C" for i in range(k))
    return dec(a) + link(k1) + dec(b) + link(k2) + dec(c)

def make_synthetic_data(n=2500, seed=0, n_scaffolds=400, interaction=(0.7, 0.5)):
    """Synthetic data with the same schema as ``load_data()`` and a known nonlinear, cross-block signal.

    Args:
        n: number of rows.
        seed: random seed (the output is fully deterministic).
        n_scaffolds: number of scaffold templates to draw molecules from (Zipf-skewed).
        interaction: coefficients of the mol x poi and cell x poi product terms.

    Returns:
        dict with the block arrays (``fingerprint`` 1024, ``descriptors`` 256, ``e3`` 10, ``cell`` 768, ``poi`` 768,
        ``assay`` 15, ``assay_time`` 1), ``smiles``, ``y`` (dict of the three task targets, NaN where undefined) and
        ``signal`` (ground-truth additive/interaction/noise components, for testing).
    """
    rng = np.random.default_rng(seed)
    # ---- molecules: skewed scaffold groups + decorations; latent mostly inherited from the scaffold
    templates = [(rng.integers(len(_SYN_RINGS)), rng.integers(1, 7), rng.integers(len(_SYN_RINGS)),
                  rng.integers(1, 7), rng.integers(len(_SYN_RINGS))) for _ in range(n_scaffolds)]
    w = 1.0 / (np.arange(n_scaffolds) + 3.0) ** 0.9
    g = rng.choice(n_scaffolds, size=n, p=w / w.sum())
    smiles = np.array([_syn_smiles(rng, templates[i]) for i in g], dtype=object)
    acyc = rng.random(n) < 0.01
    smiles[acyc] = rng.choice(_SYN_ACYCLIC, size=acyc.sum())
    m_group = rng.normal(size=(n_scaffolds, 3))
    m = 0.8 * m_group[g] + 0.6 * rng.normal(size=(n, 3))
    fp = (rng.random((n, 1024)) < 1 / (1 + np.exp(-(m @ rng.normal(0, 1.2, (3, 1024)) - 1.5)))).astype(np.float32)
    desc = (m @ rng.normal(size=(3, 256)) + rng.normal(size=(n, 5)) @ rng.normal(size=(5, 256))
            + 0.5 * rng.normal(size=(n, 256))) * rng.lognormal(0, 1, 256)
    # ---- context entities: fixed embeddings per cell line / protein (like cached text and ESM embeddings)
    n_cell, n_poi = 30, 25
    _std = lambda a: (a - a.mean(0)) / a.std(0)          # centred entity latents: products carry no linear part
    c_pool, p_pool = _std(rng.normal(size=(n_cell, 2))), _std(rng.normal(size=(n_poi, 2)))
    cell_pool = c_pool @ rng.normal(size=(2, 768)) + 0.5 * rng.normal(size=(n_cell, 768))
    poi_pool = p_pool @ rng.normal(size=(2, 768)) + 0.5 * rng.normal(size=(n_poi, 768))
    ci, pi_ = rng.integers(n_cell, size=n), rng.integers(n_poi, size=n)
    c, p = c_pool[ci], p_pool[pi_]
    e3_i = rng.choice(10, size=n, p=np.array([.45, .3, .08, .05, .04, .03, .02, .01, .01, .01]))
    as_i = rng.integers(15, size=n)
    e3 = np.eye(10, dtype=np.float32)[e3_i]
    assay = np.eye(15, dtype=np.float32)[as_i]
    t = rng.lognormal(np.log(18), 0.8, size=n)
    t[rng.random(n) < 0.4] = np.nan
    # ---- signal
    e3_eff, as_eff = 0.5 * rng.normal(size=10), 0.3 * rng.normal(size=15)
    additive = 0.9 * m[:, 0] + 0.6 * np.sin(1.5 * m[:, 1]) + 0.5 * c[:, 0] + 0.4 * p[:, 1] + e3_eff[e3_i] + as_eff[as_i]
    inter = interaction[0] * m[:, 2] * p[:, 0] + interaction[1] * c[:, 1] * p[:, 1]
    noise = 0.5 * rng.normal(size=n)
    score = additive + inter + noise
    dmax_pct = 100 / (1 + np.exp(-(1.3 * score + 0.5)))
    pdc50 = 6.5 + 0.9 * score + 0.4 * rng.normal(size=n)
    dc50 = 10.0 ** (9 - pdc50)
    dmax_pct[rng.random(n) < 0.10] = np.nan
    dc50[rng.random(n) < 0.15] = np.nan
    return {"fingerprint": fp, "descriptors": desc.astype(np.float32), "e3": e3, "cell": cell_pool[ci].astype(np.float32),
            "poi": poi_pool[pi_].astype(np.float32), "assay": assay, "assay_time": t[:, None].astype(np.float32),
            "smiles": smiles, "y": make_targets(dmax_pct, dc50),
            "signal": {"additive": additive, "interaction": inter, "noise": noise, "latent_score": score,
                       "drivers": np.column_stack([m, c, p])}}

In [ ]:
# TEST (written first): synthetic data with the same schema and a known cross-block interaction
def _test_synthetic():
    d = make_synthetic_data(n=1500, seed=0)
    n = 1500
    dims = {"fingerprint": 1024, "descriptors": 256, "e3": 10, "cell": 768, "poi": 768, "assay": 15, "assay_time": 1}
    for b in BLOCK_ORDER:
        assert d[b].shape == (n, dims[b]), (b, d[b].shape)
    assert len(d["smiles"]) == n and set(d["y"]) == {"dmax", "pdc50", "activity"}
    assert (d["e3"].sum(1) == 1).all() and (d["assay"].sum(1) == 1).all()
    assert np.isnan(d["assay_time"]).any() and not np.isnan(d["fingerprint"]).any()
    assert 0.0 <= np.nanmin(d["y"]["dmax"]) and np.nanmax(d["y"]["dmax"]) <= 1.0
    assert set(np.unique(d["y"]["activity"][~np.isnan(d["y"]["activity"])])) == {0.0, 1.0}
    # every task has enough defined rows, and some are missing (as in the real data)
    assert all(np.isfinite(d["y"][t]).sum() > 0.5 * n for t in ALL_TASKS)
    # deterministic
    d2 = make_synthetic_data(n=1500, seed=0)
    np.testing.assert_array_equal(d["fingerprint"], d2["fingerprint"])
    assert list(d["smiles"]) == list(d2["smiles"])
    # the planted signal: y = additive + interaction + noise; the cross-block product must matter and must be
    # invisible to any model that is additive in the blocks (uncorrelated with the additive part and the drivers)
    s = d["signal"]
    y = s["additive"] + s["interaction"] + s["noise"]
    np.testing.assert_allclose(y, s["latent_score"])
    assert np.var(s["interaction"]) > 0.10 * np.var(y)
    drivers = np.column_stack([s["drivers"], np.ones(n)])           # per-block latent drivers (linear terms)
    beta, *_ = np.linalg.lstsq(drivers, s["interaction"], rcond=None)
    r2_lin = 1 - np.var(s["interaction"] - drivers @ beta) / np.var(s["interaction"])
    assert r2_lin < 0.05, f"interaction should not be linearly explainable by the block drivers (R2={r2_lin:.2f})"
    # scaffold structure: parseable SMILES, many groups, a few acyclic failures
    from rdkit import Chem
    mols = [Chem.MolFromSmiles(x) for x in d["smiles"]]
    assert all(m is not None for m in mols)
    print("make_synthetic_data: OK")

_test_synthetic()

### 2.5 `BLOCKS` and the design matrix
`BLOCKS` maps each block name to its column indices in the design matrix `X` (the blocks concatenated in `BLOCK_ORDER`).
`e3`, `assay` and `assay_time` are the **small blocks**: never reduced by PCA and always passed through to every model.

In [ ]:
BLOCKS = {}   # block name -> array of column indices into the design matrix X (filled by assemble_X)

def assemble_X(data):
    """Concatenate the blocks (in ``BLOCK_ORDER``) into the design matrix X and (re)define the global ``BLOCKS``."""
    blocks = [np.asarray(data[b], dtype=np.float32) for b in BLOCK_ORDER]
    for b, arr in zip(BLOCK_ORDER, blocks):
        assert arr.ndim == 2 and arr.shape[0] == len(data["smiles"]), f"block {b} has shape {arr.shape}"
    BLOCKS.clear()
    start = 0
    for b, arr in zip(BLOCK_ORDER, blocks):
        BLOCKS[b] = np.arange(start, start + arr.shape[1])
        start += arr.shape[1]
    return np.concatenate(blocks, axis=1)

## 3. Scaffold groups and cross-validation splits
Groups are **generic Murcko scaffolds** (`MurckoScaffold.MakeScaffoldGeneric(GetScaffoldForMol(mol))`). A molecule
whose scaffold fails (acyclic molecule -> empty scaffold, or an RDKit error) gets its own group id (one per distinct
failed molecule, so identical rows still stay together). Splits are `StratifiedGroupKFold(5, shuffle=True,
random_state=repeat_seed)` stratified on the target in 5 quantile bins (the class label for the binary task), 5
repeats; the same 25 index pairs are reused for every method and cached to disk.

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

def scaffold_groups(smiles, verbose=True):
    """Integer group id per row from generic Murcko scaffolds.

    Args:
        smiles: iterable of SMILES strings.
        verbose: print how many molecules fell back to a unique group.

    Returns:
        (groups, info): int array of group ids (0..G-1) and a dict with the scaffold strings and failure counts.
    """
    smiles = list(smiles)
    key, scaf = [], {}
    for s in dict.fromkeys(smiles):
        try:
            mol = Chem.MolFromSmiles(s)
            core = MurckoScaffold.GetScaffoldForMol(mol)
            generic = MurckoScaffold.MakeScaffoldGeneric(core)
            scaf[s] = Chem.MolToSmiles(generic)
        except Exception:                      # sanitisation error / None molecule
            scaf[s] = ""
        if scaf[s] == "":                      # acyclic or failed: unique group for this molecule
            scaf[s] = f"__failed__{s}"
    ids = {k: i for i, k in enumerate(dict.fromkeys(scaf[s] for s in scaf))}
    groups = np.array([ids[scaf[s]] for s in smiles], dtype=int)
    failed_mols = [s for s in scaf if scaf[s].startswith("__failed__")]
    info = {"scaffold": scaf, "n_failed_molecules": len(failed_mols),
            "n_failed_rows": int(sum(scaf[s].startswith("__failed__") for s in smiles))}
    if verbose:
        print(f"scaffold failures (empty scaffold or RDKit error): {info['n_failed_molecules']} distinct molecules, "
              f"{info['n_failed_rows']} rows -> each failed molecule got its own group")
    return groups, info

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, GroupShuffleSplit

def _strat_labels(y, n_bins=N_STRAT_BINS):
    """Stratification variable: quantile bins of a continuous target, the labels themselves if few distinct values."""
    y = np.asarray(y)
    if len(np.unique(y)) <= n_bins:
        return np.unique(y, return_inverse=True)[1]
    return pd.qcut(pd.Series(y).rank(method="first"), n_bins, labels=False).to_numpy()

def make_splits(y, groups, n_repeats=5, n_splits=N_SPLITS, seeds=None):
    """``n_repeats`` x ``n_splits`` (train_idx, test_idx) pairs, never splitting a scaffold group."""
    seeds = list(seeds if seeds is not None else REPEAT_SEEDS)[:n_repeats]
    strat = _strat_labels(y)
    out = []
    for seed in seeds:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        out.append([(tr, te) for tr, te in cv.split(np.zeros(len(y)), strat, groups)])
    return out

def _splits_signature(y, groups, n_repeats, n_splits, seeds):
    h = hashlib.sha1()
    for a in (np.asarray(y, float), np.asarray(groups, int)):
        h.update(np.ascontiguousarray(a).tobytes())
    h.update(json.dumps([n_repeats, n_splits, list(seeds), N_STRAT_BINS]).encode())
    return h.hexdigest()

def get_splits(task, y, groups, n_repeats=5, n_splits=N_SPLITS, seeds=None, path=None):
    """Generate the splits once and cache them in ``RESULTS_DIR``; regenerate if y/groups/seeds change."""
    seeds = list(seeds if seeds is not None else REPEAT_SEEDS)[:n_repeats]
    path = Path(path or RESULTS_DIR / f"splits_{task}.pkl")
    sig = _splits_signature(y, groups, n_repeats, n_splits, seeds)
    if path.exists():
        with open(path, "rb") as f:
            cached = pickle.load(f)
        if cached["signature"] == sig:
            return cached["splits"]
        print(f"[{task}] cached splits do not match the data/seeds -> regenerating")
    splits = make_splits(y, groups, n_repeats, n_splits, seeds)
    with open(path, "wb") as f:
        pickle.dump({"signature": sig, "splits": splits, "seeds": seeds}, f)
    return splits

In [ ]:
# TEST (written first): BLOCKS / assemble_X
def _test_blocks():
    d = make_synthetic_data(n=200, seed=1)
    X = assemble_X(d)
    assert list(BLOCKS) == BLOCK_ORDER
    cols = np.concatenate([BLOCKS[b] for b in BLOCK_ORDER])
    np.testing.assert_array_equal(cols, np.arange(X.shape[1])), "blocks must tile the columns exactly once, in order"
    for b in BLOCK_ORDER:
        np.testing.assert_array_equal(np.nan_to_num(X[:, BLOCKS[b]], nan=-7), np.nan_to_num(d[b], nan=-7))
    assert set(DENSE_BLOCKS) | set(SMALL_BLOCKS) == set(BLOCK_ORDER) and not set(DENSE_BLOCKS) & set(SMALL_BLOCKS)
    assert X.dtype == np.float32 and X.shape[0] == 200
    print(f"BLOCKS: OK  X={X.shape}  " + ", ".join(f"{b}={len(BLOCKS[b])}" for b in BLOCK_ORDER))

_test_blocks()

In [ ]:
# TEST (written first): generic Murcko scaffold groups with unique ids for failures
def _test_groups():
    smi = ["c1ccccc1CCc1ccncc1",            # benzene-CC-pyridine
           "c1c(F)cccc1CCc1ccncc1C",        # same generic scaffold, other decorations/atoms -> same group
           "C1CCCCC1CCC1CCNCC1",            # saturated analogue: same *generic* scaffold -> same group
           "c1ccccc1CCCc1ccncc1",           # longer linker -> different group
           "CCO", "CCO", "CCN",             # acyclic: empty scaffold -> one unique group per failed molecule
           "this-is-not-a-smiles"]          # RDKit parse failure -> unique group
    g, info = scaffold_groups(smi, verbose=False)
    assert g[0] == g[1] == g[2] and g[3] != g[0]
    assert g[4] == g[5], "the same failed molecule must stay in one group (else identical rows could straddle train/test)"
    assert len({g[4], g[6], g[7], g[0], g[3]}) == 5
    assert info["n_failed_molecules"] == 3 and info["n_failed_rows"] == 4
    assert g.dtype.kind == "i" and g.min() == 0
    print("scaffold_groups: OK")

_test_groups()

In [ ]:
# TEST (written first): repeated stratified group splits
def _test_splits():
    rng = np.random.default_rng(0)
    n = 600
    groups = rng.choice(80, size=n, p=(lambda w: w / w.sum())(1 / (np.arange(80) + 2.0)))
    y = rng.normal(size=n) + 0.5 * (groups % 5)
    sp = make_splits(y, groups, n_repeats=2, n_splits=5, seeds=[7, 8])
    assert len(sp) == 2 and all(len(r) == 5 for r in sp)
    for r in sp:
        tests = np.concatenate([te for _, te in r])
        assert sorted(tests) == list(range(n)), "test folds of one repeat must partition the rows"
        for tr, te in r:
            assert not set(groups[tr]) & set(groups[te]), "scaffold group in both train and test"
            assert len(set(tr) & set(te)) == 0 and len(tr) + len(te) == n
    assert any(not np.array_equal(a[1], b[1]) for a, b in zip(sp[0], sp[1])), "repeats must differ"
    sp2 = make_splits(y, groups, n_repeats=2, n_splits=5, seeds=[7, 8])
    assert all(np.array_equal(a[1], b[1]) for ra, rb in zip(sp, sp2) for a, b in zip(ra, rb)), "deterministic"
    # binary target: stratification variable falls back to the labels themselves
    groups_b = rng.integers(150, size=n)                      # many small groups, labels independent of group
    yb = (rng.random(n) < 0.3).astype(float)
    spb = make_splits(yb, groups_b, n_repeats=1, n_splits=5, seeds=[3])
    rates = [yb[te].mean() for _, te in spb[0]]
    # sklearn's StratifiedGroupKFold(shuffle=True) balances strata only approximately; require both classes everywhere
    assert all(0 < r < 1 for r in rates) and max(abs(r - yb.mean()) for r in rates) < 0.12, rates
    # caching: generated once, reloaded identically, regenerated when the inputs change
    path = RESULTS_DIR / "splits_unittest.pkl"
    path.unlink(missing_ok=True)
    c1 = get_splits("unittest", y, groups, n_repeats=2, seeds=[7, 8], path=path)
    assert path.exists()
    mt = path.stat().st_mtime_ns
    c2 = get_splits("unittest", y, groups, n_repeats=2, seeds=[7, 8], path=path)
    assert path.stat().st_mtime_ns == mt and all(np.array_equal(a[1], b[1]) for ra, rb in zip(c1, c2) for a, b in zip(ra, rb))
    c3 = get_splits("unittest", y + 1e-3 * rng.normal(size=n), groups, n_repeats=2, seeds=[9, 10], path=path)
    assert path.stat().st_mtime_ns != mt
    path.unlink()
    print("make_splits / get_splits: OK")

_test_splits()

In [ ]:
def group_stats(groups, warn_frac=0.10):
    """Number of groups, largest group, singleton fractions; flags a largest group above ``warn_frac`` of the rows."""
    _, counts = np.unique(groups, return_counts=True)
    n = counts.sum()
    return {"n_rows": int(n), "n_groups": int(len(counts)), "largest_size": int(counts.max()),
            "largest_frac": float(counts.max() / n), "singleton_group_frac": float((counts == 1).mean()),
            "singleton_row_frac": float((counts == 1).sum() / n), "warn_largest": bool(counts.max() / n > warn_frac)}

def build_tasks(data, X, groups_all, tasks, n_repeats=5, results_dir=None):
    """Per-task view: rows with a defined target, their scaffold groups and the cached CV splits.

    Returns:
        dict task -> {"idx", "X", "y", "groups", "splits", "smiles"} where ``splits[r][f] = (train_idx, test_idx)``
        index into the task's own rows.
    """
    results_dir = Path(results_dir or RESULTS_DIR)
    results_dir.mkdir(parents=True, exist_ok=True)
    out = {}
    for t in tasks:
        y_all = np.asarray(data["y"][t], dtype=float)
        idx = np.flatnonzero(np.isfinite(y_all))
        y, g = y_all[idx], groups_all[idx]
        out[t] = {"idx": idx, "X": X[idx], "y": y, "groups": g, "smiles": np.asarray(data["smiles"])[idx],
                  "splits": get_splits(t, y, g, n_repeats=n_repeats, path=results_dir / f"splits_{t}.pkl")}
    return out

def assert_no_group_leakage(view):
    """No scaffold group in both train and test of any fold; test folds partition the rows within each repeat."""
    n = len(view["y"])
    for r, repeat in enumerate(view["splits"]):
        covered = np.concatenate([te for _, te in repeat])
        assert len(covered) == len(set(covered)) == n, f"repeat {r}: test folds do not partition the rows"
        for f, (tr, te) in enumerate(repeat):
            shared = set(view["groups"][tr]) & set(view["groups"][te])
            assert not shared, f"repeat {r} fold {f}: {len(shared)} scaffold group(s) in both train and test"
            assert not set(tr) & set(te), f"repeat {r} fold {f}: row in both train and test"

In [ ]:
# TEST (written first): per-task views, group statistics, and the leakage assertion
def _test_task_views():
    st_ = group_stats(np.array([0, 0, 0, 1, 2, 3]))
    assert st_["n_groups"] == 4 and st_["largest_size"] == 3 and abs(st_["largest_frac"] - 0.5) < 1e-12
    assert abs(st_["singleton_group_frac"] - 0.75) < 1e-12 and abs(st_["singleton_row_frac"] - 0.5) < 1e-12
    assert st_["warn_largest"] is True and group_stats(np.arange(50))["warn_largest"] is False

    d = make_synthetic_data(n=800, seed=2)
    X = assemble_X(d)
    g_all, _ = scaffold_groups(d["smiles"], verbose=False)
    tasks = build_tasks(d, X, g_all, TASKS_UNDER_TEST := ["dmax", "pdc50", "activity"], n_repeats=2,
                        results_dir=RESULTS_DIR / "unittest_tasks")
    for t in TASKS_UNDER_TEST:
        v = tasks[t]
        assert np.isfinite(v["y"]).all() and len(v["y"]) == len(v["X"]) == len(v["groups"]) == len(v["idx"])
        np.testing.assert_array_equal(v["X"], X[v["idx"]])
        np.testing.assert_array_equal(v["y"], d["y"][t][v["idx"]])
        assert len(v["splits"]) == 2 and all(len(r) == 5 for r in v["splits"])
        assert_no_group_leakage(v)
    # the leakage check must actually fire when a scaffold group straddles train and test
    v = dict(tasks["dmax"]); bad = [list(r) for r in v["splits"]]
    tr, te = bad[0][0]
    leaked = np.concatenate([te[:1], tr[:1]])           # not a valid split: put a train row's group into test
    bad[0][0] = (tr, np.concatenate([te, tr[v["groups"][tr] == v["groups"][tr[0]]]]))
    v["splits"] = bad
    try:
        assert_no_group_leakage(v)
    except AssertionError:
        pass
    else:
        raise RuntimeError("leakage check did not fire")
    import shutil; shutil.rmtree(RESULTS_DIR / "unittest_tasks", ignore_errors=True)
    print("task views / group stats / leakage check: OK")

_test_task_views()

In [ ]:
data = load_data()
X_all = assemble_X(data)
groups_all, group_info = scaffold_groups(data["smiles"])
print(f"\nX: {X_all.shape}   blocks: " + ", ".join(f"{b}={len(BLOCKS[b])}" for b in BLOCK_ORDER))
print(f"NaN fraction per block: " + ", ".join(f"{b}={np.isnan(X_all[:, BLOCKS[b]]).mean():.3f}" for b in BLOCK_ORDER))
print("rows with a defined target: " + ", ".join(f"{t}={int(np.isfinite(data['y'][t]).sum())}" for t in ALL_TASKS))

In [ ]:
TASK_DATA = build_tasks(data, X_all, groups_all, TASKS, n_repeats=N_REPEATS)

## 4. Split diagnostics
Group statistics up front. If the largest scaffold group exceeds ~10 % of a task's rows the folds cannot be
balanced (the whole group lands in one test fold), so a warning is printed.

In [ ]:
rows = []
for t, v in TASK_DATA.items():
    s = group_stats(v["groups"])
    fold_sizes = [len(te) for rep in v["splits"] for _, te in rep]
    rows.append({"task": t, "rows": s["n_rows"], "groups": s["n_groups"], "largest_group": s["largest_size"],
                 "largest_%": 100 * s["largest_frac"], "singleton_groups_%": 100 * s["singleton_group_frac"],
                 "singleton_rows_%": 100 * s["singleton_row_frac"], "test_size_min": min(fold_sizes),
                 "test_size_max": max(fold_sizes), "expected_test_size": s["n_rows"] // N_SPLITS})
    if s["warn_largest"]:
        print(f"WARNING [{t}]: largest scaffold group holds {100 * s['largest_frac']:.1f}% of the rows (> 10%): "
              f"fold sizes will be very unbalanced.")
diag = pd.DataFrame(rows).set_index("task").round(1)
display(diag) if "display" in globals() else print(diag)

fig, axes = plt.subplots(1, len(TASK_DATA), figsize=(4.2 * len(TASK_DATA), 3.4), squeeze=False)
for ax, (t, v) in zip(axes[0], TASK_DATA.items()):
    _, counts = np.unique(v["groups"], return_counts=True)
    ax.hist(counts, bins=np.unique(np.round(np.geomspace(1, counts.max() + 1, 25))), color="C0")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set(title=f"{t}: scaffold group sizes", xlabel="rows per scaffold group", ylabel="number of groups")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "group_sizes.png", dpi=120); plt.show()

fig, axes = plt.subplots(1, len(TASK_DATA), figsize=(4.2 * len(TASK_DATA), 3.2), squeeze=False)
for ax, (t, v) in zip(axes[0], TASK_DATA.items()):
    sizes = pd.DataFrame([{"repeat": r, "fold": f, "n_test": len(te)} for r, rep in enumerate(v["splits"])
                          for f, (_, te) in enumerate(rep)])
    sns.stripplot(data=sizes, x="fold", y="n_test", hue="repeat", palette="viridis", ax=ax, legend=False)
    ax.axhline(len(v["y"]) / N_SPLITS, color="k", ls="--", lw=1)
    ax.set(title=f"{t}: test-fold sizes (dashed = n/5)", xlabel="fold", ylabel="test rows")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "fold_sizes.png", dpi=120); plt.show()

### Leakage test
No scaffold group may appear in both the training and the test part of any fold (and each repeat's test folds must
partition the data). This runs on the actual splits used below.

In [ ]:
for t, v in TASK_DATA.items():
    assert_no_group_leakage(v)
    n_pairs = sum(len(r) for r in v["splits"])
    print(f"[{t}] OK: {n_pairs} train/test pairs, no scaffold group shared between train and test")

## 5. Method definitions and harness
### 5.1 Fold-internal preprocessing
`BlockPreprocessor` is fitted on the **training rows only** and applied to test rows:
* dense blocks (`fingerprint`, `descriptors`, `cell`, `poi`): mean-impute (train mean) -> `StandardScaler` -> divide
  by `sqrt(block_dim)` (so no block dominates by width) -> optional `PCA(k)`, `k = min(pca_k, n_train - 1, block_dim)`;
* `reduce_blocks` / `scale_blocks` restrict PCA / standardisation to some of the dense blocks (default: all). M7 uses this to
  reduce only `cell` and `poi`, leaving the Morgan bits and RDKit descriptors as raw (mean-imputed) columns;
* small blocks (`e3`, `assay`, `assay_time`): never reduced; one-hots pass through unscaled, `assay_time` is
  median-imputed (train median) and standardised (train mean/std).

Targets are standardised on the training fold inside each estimator and inverted before scoring.

Inner CV loops (hyper-parameter tuning, out-of-fold stacking, Platt calibration) use the scaler/PCA fitted on the *whole outer
training fold* instead of refitting them per inner fold: they are unsupervised and see no outer-test row, so nothing leaks
into the outer scores; it only makes the inner validation scores marginally optimistic.

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

class BlockPreprocessor(BaseEstimator):
    """Per-block imputation, scaling by 1/sqrt(width) and optional PCA, fitted on the rows passed to ``fit`` only.

    Args:
        blocks: dict block name -> column indices of X (default: the global ``BLOCKS``).
        reduce: apply ``PCA`` to the dense blocks.
        pca_k: cap on the number of components; the effective k is ``min(pca_k, n_train - 1, block_dim)``.
        dense: names of the blocks that are imputed, scaled and (if ``reduce``) reduced; the others are passed through.
        random_state: seed of the (randomised) PCA solver.
        reduce_blocks: dense blocks that ``reduce`` applies PCA to (default ``None``: all of them).
        scale_blocks: dense blocks that are standardised and divided by ``sqrt(block_dim)`` (default ``None``: all of them);
            the others are only mean-imputed and keep their raw units (for scale-free models such as trees).
    """
    def __init__(self, blocks=None, reduce=False, pca_k=MAX_PCA, dense=tuple(DENSE_BLOCKS), random_state=0,
                 reduce_blocks=None, scale_blocks=None):
        self.blocks, self.reduce, self.pca_k, self.dense, self.random_state = blocks, reduce, pca_k, dense, random_state
        self.reduce_blocks, self.scale_blocks = reduce_blocks, scale_blocks

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=np.float64)
        self.blocks_ = {b: np.asarray(i) for b, i in (self.blocks or BLOCKS).items()}
        self.steps_, self.dims_ = {}, {}
        reduced = self._select(self.reduce_blocks, "reduce_blocks") if self.reduce else set()
        scaled = self._select(self.scale_blocks, "scale_blocks")
        for b, idx in self.blocks_.items():
            Xb = X[:, idx]
            if b in self.dense:
                imp = SimpleImputer(strategy="mean", keep_empty_features=True).fit(Xb)
                sc = StandardScaler().fit(imp.transform(Xb)) if b in scaled else None
                width = np.sqrt(Xb.shape[1]) if b in scaled else 1.0
                pca = None
                if b in reduced:
                    k = int(min(self.pca_k, X.shape[0] - 1, Xb.shape[1]))
                    Z = imp.transform(Xb)
                    pca = PCA(n_components=k, random_state=self.random_state).fit(Z if sc is None else sc.transform(Z) / width)
                self.steps_[b] = (imp, sc, width, pca)
                self.dims_[b] = pca.n_components_ if pca is not None else Xb.shape[1]
            elif b == "assay_time":
                imp = SimpleImputer(strategy="median", keep_empty_features=True).fit(Xb)
                sc = StandardScaler().fit(imp.transform(Xb))
                self.steps_[b] = (imp, sc, 1.0, None)
                self.dims_[b] = Xb.shape[1]
            else:                                            # one-hots and other small blocks: unscaled pass-through
                self.steps_[b] = None
                self.dims_[b] = Xb.shape[1]
        return self

    def _select(self, names, arg):
        """The dense blocks named by ``reduce_blocks`` / ``scale_blocks`` (``None`` = all); unknown names are an error."""
        dense = set(self.dense)
        chosen = dense if names is None else set(names)
        if not chosen <= dense:
            raise ValueError(f"{arg} must be dense blocks {sorted(dense)}, got {sorted(chosen - dense)}")
        return chosen

    def transform(self, X):
        """Dict block -> processed array (float64) for the rows of X."""
        X = np.asarray(X, dtype=np.float64)
        out = {}
        for b, idx in self.blocks_.items():
            Xb = X[:, idx]
            step = self.steps_[b]
            if step is None:
                out[b] = np.nan_to_num(Xb, nan=0.0)
                continue
            imp, sc, width, pca = step
            Z = imp.transform(Xb)
            if sc is not None:
                Z = sc.transform(Z) / width
            out[b] = pca.transform(Z) if pca is not None else Z
        return out

    @staticmethod
    def concat(Z, names=None):
        names = list(names or [b for b in BLOCK_ORDER if b in Z])
        return np.concatenate([Z[b] for b in names], axis=1)

In [ ]:
# TEST (written first): the preprocessor is fitted on train only and never looks at test rows
def _test_preproc():
    d = make_synthetic_data(n=700, seed=3)
    X = assemble_X(d)
    tr, te = np.arange(0, 500), np.arange(500, 700)

    pre = BlockPreprocessor(reduce=False).fit(X[tr])
    Z = pre.transform(X[te])
    # (1) statistics are the train statistics: standardised train columns have mean 0 / std 1 / sqrt(dim)
    Ztr = pre.transform(X[tr])
    for b in ["cell", "poi"]:
        np.testing.assert_allclose(Ztr[b].mean(0), 0, atol=1e-5)
        np.testing.assert_allclose(Ztr[b].std(0) * np.sqrt(Ztr[b].shape[1]), 1, atol=1e-3)
        assert 0.95 < (Ztr[b] ** 2).sum(1).mean() < 1.05, "each dense block has unit squared norm on average"
    # (2) test rows are transformed row-wise with train statistics: independent of the other test rows
    np.testing.assert_allclose(pre.transform(X[te[:1]])["cell"], Z["cell"][:1], atol=1e-6)
    X2 = X.copy(); X2[np.ix_(te, BLOCKS["cell"])] += 100.0       # perturb the test cells only
    np.testing.assert_allclose(BlockPreprocessor(reduce=False).fit(X2[tr]).transform(X2[tr])["cell"], Ztr["cell"], atol=1e-6)
    # (3) assay_time: train median imputation, then train standardisation
    t = X[:, BLOCKS["assay_time"][0]]
    med, mu_, sd_ = np.nanmedian(t[tr]), None, None
    filled = np.where(np.isnan(t[tr]), med, t[tr]); mu_, sd_ = filled.mean(), filled.std()
    exp = (np.where(np.isnan(t[te]), med, t[te]) - mu_) / sd_
    np.testing.assert_allclose(Z["assay_time"][:, 0], exp, atol=1e-5)
    # (4) small blocks pass through: one-hots untouched
    np.testing.assert_array_equal(Z["e3"], X[te][:, BLOCKS["e3"]]); np.testing.assert_array_equal(Z["assay"], X[te][:, BLOCKS["assay"]])
    # (5) NaN in a dense block is imputed with the train mean (-> 0 after standardisation)
    Xn = X.copy(); Xn[te[0], BLOCKS["poi"]] = np.nan
    np.testing.assert_allclose(BlockPreprocessor(reduce=False).fit(Xn[tr]).transform(Xn[te[:1]])["poi"], 0, atol=1e-6)
    # (6) PCA: k = min(pca_k, n_train - 1, block_dim), fitted on train only
    pca = BlockPreprocessor(reduce=True, pca_k=64).fit(X[tr])
    Zp = pca.transform(X[te])
    assert Zp["fingerprint"].shape[1] == 64 and Zp["cell"].shape[1] == 64 and Zp["e3"].shape[1] == 10
    tiny = BlockPreprocessor(reduce=True, pca_k=64).fit(X[:30])
    assert tiny.transform(X[:5])["cell"].shape[1] == 29, "k must not exceed n_train - 1"
    assert BlockPreprocessor(reduce=True, pca_k=16).fit(X[tr]).transform(X[te])["descriptors"].shape[1] == 16
    assert pca.dims_["assay_time"] == 1 and pca.dims_["descriptors"] == 64
    # fit on the train rows only: PCA basis does not move when only test rows change
    pca2 = BlockPreprocessor(reduce=True, pca_k=8).fit(X2[tr])
    pca1 = BlockPreprocessor(reduce=True, pca_k=8).fit(X[tr])
    np.testing.assert_allclose(pca1.transform(X[te])["descriptors"], pca2.transform(X[te])["descriptors"], atol=1e-5)
    # (7) selective reduction: PCA (and scaling) only for the listed blocks, the other dense blocks keep their raw columns
    sel = BlockPreprocessor(reduce=True, pca_k=8, reduce_blocks=("cell", "poi"), scale_blocks=("cell", "poi")).fit(X[tr])
    Zs = sel.transform(X[te])
    assert Zs["cell"].shape[1] == 8 and Zs["poi"].shape[1] == 8 and sel.dims_["cell"] == sel.dims_["poi"] == 8
    for b in ("fingerprint", "descriptors"):
        assert Zs[b].shape[1] == len(BLOCKS[b]) and sel.dims_[b] == len(BLOCKS[b])
        raw = X[te][:, BLOCKS[b]]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mean_tr = np.nan_to_num(np.nanmean(X[tr][:, BLOCKS[b]], axis=0))
        np.testing.assert_allclose(Zs[b], np.where(np.isnan(raw), mean_tr, raw), atol=1e-6)   # raw values; NaN -> train mean only
    # the reduced blocks are exactly what the all-blocks preprocessor gives for them (same scaling and PCA seed)
    allb = BlockPreprocessor(reduce=True, pca_k=8).fit(X[tr]).transform(X[te])
    np.testing.assert_allclose(np.abs(Zs["cell"]), np.abs(allb["cell"]), atol=1e-6)         # PCA sign is arbitrary
    # defaults are unchanged: ``reduce_blocks=None`` reduces every dense block; unknown names are an error, not a silent no-op
    assert all(BlockPreprocessor(reduce=True, pca_k=8).fit(X[tr]).dims_[b] == 8 for b in DENSE_BLOCKS)
    for bad in (dict(reduce_blocks=("cel",)), dict(scale_blocks=("e3",))):
        try:
            BlockPreprocessor(reduce=True, **bad).fit(X[tr])
        except ValueError:
            pass
        else:
            raise AssertionError(f"{bad} must be rejected")
    print("BlockPreprocessor: OK")

_test_preproc()

### 5.2 Estimator base class, M1 (ridge) and M2 (stacking)
Every method is a scikit-learn style estimator (`get_params`/`clone` work) with `fit(X, y, groups)` and `predict(X)`.
The base class fits the `BlockPreprocessor` on the training rows, standardises the target on the training fold and
inverts it in `predict`; for the binary task `predict` returns calibrated probabilities (Platt scaling on scaffold-grouped
inner out-of-fold scores for M1-M4; a BCE head for M5/M6; a classifier for M7).

* **M1**: ridge on the scaled concatenation of all blocks (no PCA), `RidgeCV` with 15 alphas from 1e-2 to 1e5, tuned with
  scaffold-grouped `GroupKFold(3)` on the training fold.
* **M2**: one ridge per block (fingerprint, descriptors, cell, poi) plus one "metadata" expert on the small blocks together;
  out-of-fold predictions from scaffold-grouped `GroupKFold(5)`, combined with `LinearRegression(positive=True)` (non-negative
  least squares + intercept); experts are refit on the full training fold. The learned weights are stored per fold.

In [ ]:
from sklearn.linear_model import RidgeCV, Ridge, LinearRegression, LogisticRegression

RIDGE_ALPHAS = np.logspace(-2, 5, 15)     # 1e-2 ... 1e5, log-spaced (M1, and the per-expert alphas of M2)

def slice_Z(Z, idx):
    """Rows ``idx`` of every block of a processed-block dict."""
    return {b: a[idx] for b, a in Z.items()}

def inner_group_splits(groups, n_splits=INNER_FOLDS):
    """Scaffold-grouped inner CV splits (``GroupKFold``) on the training fold."""
    n_splits = int(min(n_splits, len(np.unique(groups))))
    return list(GroupKFold(n_splits=n_splits).split(np.zeros(len(groups)), groups=groups))

class FusionBase(BaseEstimator, RegressorMixin):
    """Common harness for all fusion methods: fold-internal preprocessing, target standardisation, binary handling.

    Sub-classes implement ``_fit_model(Z, ys, groups, y_raw, hyper=None) -> (model, hyper)`` (``hyper`` given = reuse
    those hyper-parameters instead of searching) and ``_predict_model(model, Z) -> scores`` in standardised-target units.
    ``fit(X, y, groups)`` takes the *training rows only*; ``predict`` returns values in original target units
    (``task_type='binary'``: calibrated probability of the positive class).

    Args:
        task_type: ``'regression'`` or ``'binary'``.
        blocks: dict block name -> column indices (default: global ``BLOCKS``).
        random_state: seed for any stochastic component.
    """
    NATIVE_BINARY = False     # True when the model itself outputs probabilities (BCE head, classifier)
    REDUCE = False            # whether the preprocessing applies PCA to the dense blocks
    PCA_K = MAX_PCA

    def __init__(self, task_type="regression", blocks=None, random_state=0):
        self.task_type, self.blocks, self.random_state = task_type, blocks, random_state

    def _make_preprocessor(self):
        return BlockPreprocessor(blocks=self.blocks, reduce=self.REDUCE, pca_k=self.PCA_K, random_state=self.random_state)

    def fit(self, X, y, groups=None):
        y = np.asarray(y, dtype=float)
        groups = np.arange(len(y)) if groups is None else np.asarray(groups)
        self.y_mean_, self.y_std_ = float(y.mean()), float(y.std()) or 1.0
        self.pre_ = self._make_preprocessor().fit(X)
        Z = self.pre_.transform(X)
        ys = (y - self.y_mean_) / self.y_std_
        self.model_, self.hyper_ = self._fit_model(Z, ys, groups, y)
        if self.task_type == "binary" and not self.NATIVE_BINARY:
            oof = self._oof_scores(Z, ys, groups)
            self.platt_ = LogisticRegression(C=1e4).fit(oof[:, None], y.astype(int))
        return self

    def _oof_scores(self, Z, ys, groups):
        """Scaffold-grouped inner out-of-fold scores with frozen hyper-parameters (training data for Platt scaling)."""
        oof = np.zeros(len(ys))
        for a, b in inner_group_splits(groups):
            model, _ = self._fit_model(slice_Z(Z, a), ys[a], groups[a], None, hyper=self.hyper_)
            oof[b] = self._predict_model(model, slice_Z(Z, b))
        return oof

    def predict(self, X):
        s = self._predict_model(self.model_, self.pre_.transform(X))
        if self.task_type == "binary":
            return np.clip(s, 0.0, 1.0) if self.NATIVE_BINARY else self.platt_.predict_proba(s[:, None])[:, 1]
        return s * self.y_std_ + self.y_mean_

In [ ]:
class M1Ridge(FusionBase):
    """M1: ridge on the scaled concatenation of all blocks (no PCA). ``RidgeCV`` over ``RIDGE_ALPHAS`` with
    scaffold-grouped ``GroupKFold(3)`` on the training fold. Inner grid: 15 alphas x 3 folds."""

    def _fit_model(self, Z, ys, groups, y_raw, hyper=None):
        A = self.pre_.concat(Z)
        if hyper is None:
            m = RidgeCV(alphas=RIDGE_ALPHAS, cv=inner_group_splits(groups), scoring="neg_mean_squared_error").fit(A, ys)
            return m, {"alpha": float(m.alpha_)}
        return Ridge(alpha=hyper["alpha"]).fit(A, ys), hyper

    def _predict_model(self, model, Z):
        return model.predict(self.pre_.concat(Z))


class M2Stack(FusionBase):
    """M2: stacking of five ridge experts (fingerprint, descriptors, cell, poi, metadata = small blocks together).

    Each expert's alpha is chosen by scaffold-grouped ``GroupKFold(3)`` on the training fold; out-of-fold expert
    predictions come from scaffold-grouped ``GroupKFold(5)`` (alphas frozen); the combiner is
    ``LinearRegression(positive=True, fit_intercept=True)`` on those OOF predictions (non-negative weights; the sum-to-one
    shares are stored as ``weight_shares_``). Experts are then refit on the full training fold.
    """
    EXPERTS = {"fingerprint": ["fingerprint"], "descriptors": ["descriptors"], "cell": ["cell"], "poi": ["poi"],
               "metadata": SMALL_BLOCKS}

    def _expert_matrix(self, Z, name):
        return self.pre_.concat(Z, self.EXPERTS[name])

    def _fit_model(self, Z, ys, groups, y_raw, hyper=None):
        names = list(self.EXPERTS)
        inner = inner_group_splits(groups)
        alphas = dict(hyper["alphas"]) if hyper else {}
        for nm in names:
            if nm not in alphas:
                A = self._expert_matrix(Z, nm)
                alphas[nm] = float(RidgeCV(alphas=RIDGE_ALPHAS, cv=inner, scoring="neg_mean_squared_error").fit(A, ys).alpha_)
        oof_splits = inner_group_splits(groups, n_splits=5)
        P = np.zeros((len(ys), len(names)))
        for j, nm in enumerate(names):
            A = self._expert_matrix(Z, nm)
            for a, b in oof_splits:
                P[b, j] = Ridge(alpha=alphas[nm]).fit(A[a], ys[a]).predict(A[b])
        comb = LinearRegression(positive=True, fit_intercept=True).fit(P, ys)
        experts = {nm: Ridge(alpha=alphas[nm]).fit(self._expert_matrix(Z, nm), ys) for nm in names}
        self.expert_names_, self.oof_splits_ = names, oof_splits
        self.weights_ = comb.coef_.copy()
        self.weight_shares_ = self.weights_ / max(self.weights_.sum(), 1e-12)
        self._stack_oof_ = comb.predict(P)
        return {"experts": experts, "comb": comb}, {"alphas": alphas}

    def _predict_model(self, model, Z):
        P = np.column_stack([model["experts"][nm].predict(self._expert_matrix(Z, nm)) for nm in self.EXPERTS])
        return model["comb"].predict(P)

    def _oof_scores(self, Z, ys, groups):
        return self._stack_oof_      # the stack's own out-of-fold predictions are the calibration scores

In [ ]:
# helper shared by the estimator tests: one scaffold-grouped train/test split of synthetic data
def _synthetic_fold(n=900, seed=5, task="dmax"):
    d = make_synthetic_data(n=n, seed=seed)
    X = assemble_X(d)
    g, _ = scaffold_groups(d["smiles"], verbose=False)
    ok = np.isfinite(d["y"][task])
    X, y, g = X[ok], d["y"][task][ok], g[ok]
    tr, te = make_splits(y, g, n_repeats=1, seeds=[11])[0][0]
    return X, y, g, tr, te

In [ ]:
# TEST (written first): estimator API, M1 ridge and M2 stacking
def _test_m1_m2():
    from sklearn.metrics import r2_score, roc_auc_score
    X, y, g, tr, te = _synthetic_fold()
    for cls in (M1Ridge, M2Stack):
        est = cls()
        assert clone(est).get_params() == est.get_params(), "sklearn clone/get_params must work"
        est.fit(X[tr], y[tr], groups=g[tr])
        p = est.predict(X[te])
        assert p.shape == (len(te),) and np.isfinite(p).all()
        r2 = r2_score(y[te], p)
        assert r2 > 0.15, f"{cls.__name__} should beat the mean predictor on the synthetic signal (R2={r2:.3f})"
        np.testing.assert_allclose(est.predict(X[te]), p)                       # predict is pure
        p2 = cls().fit(X[tr], y[tr], groups=g[tr]).predict(X[te])
        np.testing.assert_allclose(p2, p, atol=1e-8)                            # deterministic
    m1 = M1Ridge().fit(X[tr], y[tr], groups=g[tr])
    assert m1.hyper_["alpha"] in set(RIDGE_ALPHAS) and RIDGE_ALPHAS[0] == 1e-2 and RIDGE_ALPHAS[-1] == 1e5
    # M2: five experts, non-negative weights, OOF built on scaffold-grouped GroupKFold(5)
    m2 = M2Stack().fit(X[tr], y[tr], groups=g[tr])
    assert m2.expert_names_ == ["fingerprint", "descriptors", "cell", "poi", "metadata"]
    assert m2.weights_.shape == (5,) and (m2.weights_ >= 0).all() and m2.weights_.sum() > 0
    assert abs(m2.weight_shares_.sum() - 1) < 1e-9
    assert len(m2.oof_splits_) == 5
    for a, b in m2.oof_splits_:
        assert not set(g[tr][a]) & set(g[tr][b]), "OOF folds of the stack must be scaffold-grouped"
    assert sorted(np.concatenate([b for _, b in m2.oof_splits_])) == list(range(len(tr)))
    # binary task: predict returns calibrated probabilities
    Xb, yb, gb, trb, teb = _synthetic_fold(task="activity")
    for cls in (M1Ridge, M2Stack):
        pb = cls(task_type="binary").fit(Xb[trb], yb[trb], groups=gb[trb]).predict(Xb[teb])
        assert ((pb >= 0) & (pb <= 1)).all() and roc_auc_score(yb[teb], pb) > 0.6, cls.__name__
    print(f"M1/M2: OK  (M1 R2={r2_score(y[te], m1.predict(X[te])):.3f}, M2 R2={r2_score(y[te], m2.predict(X[te])):.3f}, "
          f"M2 weights={np.round(m2.weight_shares_, 2)})")

_test_m1_m2()

### 5.3 M3 / M4: additive-kernel GP, with and without cross-block product kernels
Dense blocks are PCA-reduced (`k = min(64, n_train - 1, dim)`, exposed as `pca_k`). The prior covariance is
`sum_b s_b * RBF_b` (one lengthscale per block, no ARD) `+ linear kernel on the small blocks + white noise`.
**M4** adds the mechanistically plausible products `s * (RBF_fp + RBF_desc) * RBF_poi` (molecule x protein),
`... * RBF_cell` (molecule x cell line) and `RBF_poi * RBF_cell`, reusing the block lengthscales.

Hyper-parameters are fitted by marginal likelihood with **3 random restarts** (Adam) on a random subset of at most
`max_hyper_points` training rows (an exact GP needs O(n^3) per step, which is not affordable for 25 folds x 3 tasks
x 2 methods); the predictive mean then uses the exact GP on *all* training rows with those hyper-parameters.
`kernel_weights_` stores each term's share of the prior variance (diagnostic).

In [ ]:
import torch, gpytorch
warnings.filterwarnings("ignore", message=".*added jitter.*")          # GPyTorch Cholesky jitter notices

class _AdditiveGP(gpytorch.models.ExactGP):
    """Exact GP whose covariance is a sum of block RBFs, a small-block linear kernel and optional products."""
    def __init__(self, Ztr, ytr, likelihood, cols, small_cols, interactions):
        super().__init__(Ztr, ytr, likelihood)
        K = gpytorch.kernels
        self.mean_module = gpytorch.means.ConstantMean()
        self.rbf = torch.nn.ModuleDict({b: K.RBFKernel(active_dims=torch.as_tensor(c)) for b, c in cols.items()})
        terms = {b: K.ScaleKernel(self.rbf[b]) for b in cols}
        terms["small_linear"] = K.LinearKernel(active_dims=torch.as_tensor(small_cols))
        if interactions:
            mol = self.rbf["fingerprint"] + self.rbf["descriptors"]
            terms["prod:mol*poi"] = K.ScaleKernel(mol * self.rbf["poi"])
            terms["prod:mol*cell"] = K.ScaleKernel(mol * self.rbf["cell"])
            terms["prod:poi*cell"] = K.ScaleKernel(self.rbf["poi"] * self.rbf["cell"])
        self.terms = torch.nn.ModuleDict({k.replace(":", "_").replace("*", "x"): v for k, v in terms.items()})
        self.term_names = list(terms)
        self.covar_module = K.AdditiveKernel(*terms.values())

    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(self.mean_module(x), self.covar_module(x))


class GPFusion(FusionBase):
    """M3 (``interactions=False``) / M4 (``interactions=True``): additive-kernel GP via GPyTorch.

    Args:
        interactions: add the three cross-block product kernels (M4).
        pca_k: cap on PCA components per dense block.
        n_restarts: random restarts of the marginal-likelihood optimisation.
        n_iter: Adam steps per restart.
        lr: Adam learning rate.
        max_hyper_points: rows used to fit the hyper-parameters (the exact GP then uses all training rows).
    """
    REDUCE = True

    def __init__(self, task_type="regression", blocks=None, random_state=0, interactions=False, pca_k=MAX_PCA,
                 n_restarts=3, n_iter=60, lr=0.1, max_hyper_points=1200):
        super().__init__(task_type, blocks, random_state)
        self.interactions, self.pca_k, self.n_restarts = interactions, pca_k, n_restarts
        self.n_iter, self.lr, self.max_hyper_points = n_iter, lr, max_hyper_points

    @property
    def PCA_K(self):
        return self.pca_k

    def _layout(self):
        cols, start = {}, 0
        for b in BLOCK_ORDER:
            cols[b] = np.arange(start, start + self.pre_.dims_[b]); start += self.pre_.dims_[b]
        small = np.concatenate([cols[b] for b in SMALL_BLOCKS])
        return {b: cols[b] for b in DENSE_BLOCKS}, small

    def _build(self, A, ys):
        cols, small = self._layout()
        lik = gpytorch.likelihoods.GaussianLikelihood(noise_constraint=gpytorch.constraints.GreaterThan(1e-3))
        gp = _AdditiveGP(torch.as_tensor(A), torch.as_tensor(ys), lik, cols, small, self.interactions)
        return gp.double(), lik.double()

    def _init_hypers(self, gp, lik, rng):
        n_terms = len(gp.term_names)
        for b in gp.rbf:
            gp.rbf[b].lengthscale = float(np.exp(rng.normal(0.0, 0.4)))
        for name, term in zip(gp.term_names, gp.terms.values()):
            v = float(np.exp(rng.normal(0.0, 0.4)) / n_terms)
            if hasattr(term, "outputscale"):
                term.outputscale = v
            else:
                term.variance = v
        lik.noise = 0.5

    def _fit_model(self, Z, ys, groups, y_raw, hyper=None):
        A = self.pre_.concat(Z)
        if hyper is None:
            rng = np.random.default_rng(self.random_state)
            sub = np.arange(len(ys)) if len(ys) <= self.max_hyper_points else np.sort(
                rng.choice(len(ys), self.max_hyper_points, replace=False))
            best = (np.inf, None)
            for r in range(self.n_restarts):
                torch.manual_seed(self.random_state * 1000 + r)
                gp, lik = self._build(A[sub], ys[sub])
                self._init_hypers(gp, lik, np.random.default_rng(self.random_state * 1000 + r))
                gp.train(); lik.train()
                opt = torch.optim.Adam(gp.parameters(), lr=self.lr)
                mll = gpytorch.mlls.ExactMarginalLogLikelihood(lik, gp)
                for _ in range(self.n_iter):
                    opt.zero_grad()
                    loss = -mll(gp(gp.train_inputs[0]), gp.train_targets)
                    loss.backward(); opt.step()
                with torch.no_grad():
                    final = -mll(gp(gp.train_inputs[0]), gp.train_targets).item()
                if np.isfinite(final) and final < best[0]:
                    best = (final, {k: v.detach().clone() for k, v in gp.state_dict().items()})
            if best[1] is None:
                raise RuntimeError("GP marginal-likelihood optimisation diverged in every restart")
            hyper = {"state": best[1], "neg_mll": best[0]}
        gp, lik = self._build(A, ys)
        gp.load_state_dict(hyper["state"])
        gp.eval(); lik.eval()
        self._report(gp, lik, A)
        return (gp, lik), hyper

    def _report(self, gp, lik, A):
        with torch.no_grad():
            At = torch.as_tensor(A[: min(len(A), 400)])
            shares = {nm: float(t(At).diag().mean()) for nm, t in zip(gp.term_names, gp.terms.values())}
            tot = sum(shares.values()) or 1.0
            self.kernel_weights_ = {k: v / tot for k, v in shares.items()}
            self.kernel_report_ = {**{b: float(gp.rbf[b].lengthscale) for b in gp.rbf},
                                   "small_linear": shares["small_linear"], "noise": float(lik.noise)}

    def _predict_model(self, model, Z):
        gp, lik = model
        with torch.no_grad(), gpytorch.settings.max_cholesky_size(100000), gpytorch.settings.skip_posterior_variances():
            return gp(torch.as_tensor(self.pre_.concat(Z))).mean.numpy()

In [ ]:
# TEST (written first): additive-kernel GP (M3) and GP with product kernels (M4)
def _test_gp():
    import time as _t
    from sklearn.metrics import r2_score
    # strong planted cross-block interaction (mol x poi, cell x poi) so M4 has something to find
    d = make_synthetic_data(n=1800, seed=21, interaction=(1.6, 1.2))
    X = assemble_X(d); g, _ = scaffold_groups(d["smiles"], verbose=False)
    y = d["y"]["dmax"]; ok = np.isfinite(y)
    X, y, g = X[ok], y[ok], g[ok]
    tr, te = make_splits(y, g, n_repeats=1, seeds=[4])[0][0]
    res = {}
    for name, inter in (("M3", False), ("M4", True)):
        t0 = _t.time()
        est = GPFusion(interactions=inter, max_hyper_points=600, n_restarts=3, n_iter=40).fit(X[tr], y[tr], groups=g[tr])
        p = est.predict(X[te]); res[name] = r2_score(y[te], p)
        assert p.shape == (len(te),) and np.isfinite(p).all()
        w = est.kernel_weights_
        assert all(v >= 0 for v in w.values()) and abs(sum(w.values()) - 1) < 1e-6
        assert {"fingerprint", "descriptors", "cell", "poi", "small_linear", "noise"} <= set(est.kernel_report_)
        assert (any(k.startswith("prod:") for k in w)) == inter
        if inter:
            assert {"prod:mol*poi", "prod:mol*cell", "prod:poi*cell"} == {k for k in w if k.startswith("prod:")}
        np.testing.assert_allclose(est.predict(X[te]), p)                       # predict is pure
        print(f"  {name}: R2={res[name]:.3f}  fit+predict {_t.time() - t0:.0f}s  weights={ {k: round(v, 2) for k, v in w.items()} }")
    assert res["M3"] > 0.15, res
    assert res["M4"] > res["M3"] + 0.02, f"M4 should recover the planted cross-block interaction: {res}"
    # same seed -> same fit
    a = GPFusion(max_hyper_points=300, n_restarts=2, n_iter=15, random_state=3).fit(X[tr[:500]], y[tr[:500]], g[tr[:500]]).predict(X[te])
    b = GPFusion(max_hyper_points=300, n_restarts=2, n_iter=15, random_state=3).fit(X[tr[:500]], y[tr[:500]], g[tr[:500]]).predict(X[te])
    np.testing.assert_allclose(a, b, atol=1e-6)
    # binary: calibrated probabilities
    from sklearn.metrics import roc_auc_score
    yb = d["y"]["activity"]; okb = np.isfinite(yb)
    Xb, yb, gb = assemble_X(d)[okb], yb[okb], scaffold_groups(d["smiles"][okb], verbose=False)[0]
    trb, teb = make_splits(yb, gb, n_repeats=1, seeds=[4])[0][0]
    pb = GPFusion(task_type="binary", max_hyper_points=400, n_restarts=2, n_iter=25).fit(Xb[trb], yb[trb], gb[trb]).predict(Xb[teb])
    assert ((pb >= 0) & (pb <= 1)).all() and roc_auc_score(yb[teb], pb) > 0.6
    print("GP (M3/M4): OK")

_test_gp()

### 5.4 M5 (low-rank multimodal fusion) and M6 (FiLM)
Both are small PyTorch models trained with **AdamW + weight decay**, **early stopping on a scaffold-grouped validation
split** (20 % of the training groups, `GroupShuffleSplit`), and a **fixed seed per fold** (`random_state`).
Inner tuning: `GroupKFold(3)` on the training fold over the small grids below (6 and 4 configurations).

* **M5 LMF**: dense blocks PCA-reduced to 32 components, the small blocks form a fifth "meta" modality; each
  modality gets a constant 1 appended, is projected with rank-`r` factors `W_m[i]` into `fusion_dim`, projections are
  multiplied element-wise across modalities, summed over the rank, then a linear head. Factors start at "projection ~ 1"
  (the constant-1 row is initialised to 1) so the product begins as a smooth multiplicative interaction.
* **M6 FiLM**: primary = molecular blocks (fingerprint + descriptors, scaled) reduced by PCA to `ctx_dim`; context =
  PCA(`cell`, `poi`) to the same dimension plus the unreduced small blocks (`e3`, `assay`, `assay_time`);
  one linear layer maps context to `(gamma, beta)` and `h = (1 + gamma) * h_mol + beta` (zero-initialised so the
  modulation starts as the identity), followed by a one-hidden-layer MLP head.
* Binary task: BCE-with-logits head, `predict` returns `sigmoid(logit)`.

In [ ]:
import torch.nn as nn

def lmf_param_count(dims, rank, fusion_dim):
    """Parameters of ``LMFNet``: rank-r factors of every (modality + constant 1) plus the linear head."""
    return sum(rank * (d + 1) * fusion_dim for d in dims) + fusion_dim + 1

class LMFNet(nn.Module):
    """Low-rank multimodal fusion (Liu et al., 2018): ``sum_r prod_m (z_m^+ W_m[r])`` followed by a linear head."""
    def __init__(self, dims, rank, fusion_dim, dropout=0.0):
        super().__init__()
        self.factors = nn.ParameterList()
        for d in dims:
            W = torch.randn(rank, d + 1, fusion_dim) * (0.1 / np.sqrt(d))
            W[:, -1, :] = 1.0                                   # the appended constant 1 starts the projection at ~1
            self.factors.append(nn.Parameter(W))
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(fusion_dim, 1)
        nn.init.normal_(self.head.weight, std=0.05); nn.init.zeros_(self.head.bias)

    def forward(self, zs):
        ones = torch.ones(zs[0].shape[0], 1, dtype=zs[0].dtype)
        fused = None
        for z, W in zip(zs, self.factors):
            proj = torch.einsum("bd,rdh->brh", torch.cat([z, ones], dim=1), W)
            fused = proj if fused is None else fused * proj
        return self.head(self.drop(fused.sum(dim=1))).squeeze(-1)

class FiLMNet(nn.Module):
    """FiLM: context -> (gamma, beta); ``h = (1 + gamma) * h_mol + beta``; MLP head with one hidden layer."""
    def __init__(self, d_mol, d_ctx, hidden, dropout=0.1):
        super().__init__()
        self.film = nn.Linear(d_ctx, 2 * d_mol)
        nn.init.zeros_(self.film.weight); nn.init.zeros_(self.film.bias)      # identity at initialisation
        self.head = nn.Sequential(nn.Linear(d_mol, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, 1))

    def forward(self, mol, ctx):
        gamma, beta = self.film(ctx).chunk(2, dim=-1)
        return self.head((1 + gamma) * mol + beta).squeeze(-1)

def train_net(net, mods, target, groups, binary, lr, weight_decay, max_epochs, patience, batch_size, seed):
    """AdamW with early stopping on a scaffold-grouped validation split.

    Args:
        net: module taking a list of modality tensors.
        mods: list of float arrays (n, d_m).
        target: 1-D array (standardised y, or 0/1 labels when ``binary``).
        groups: scaffold groups of the rows.
        binary: BCE-with-logits loss instead of MSE.
        seed: seeds the split and the mini-batch order.

    Returns:
        (net with the best-validation weights, validation row indices, epochs run)
    """
    split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_i, va_i = next(split.split(np.zeros(len(target)), groups=groups))
    ten = [torch.as_tensor(m, dtype=torch.float32) for m in mods]
    y = torch.as_tensor(target, dtype=torch.float32)
    lossf = nn.BCEWithLogitsLoss() if binary else nn.MSELoss()
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=weight_decay)
    gen = torch.Generator().manual_seed(seed)
    best, best_state, bad, ep = np.inf, None, 0, 0
    for ep in range(1, max_epochs + 1):
        net.train()
        perm = torch.as_tensor(tr_i)[torch.randperm(len(tr_i), generator=gen)]
        for s in range(0, len(perm), batch_size):
            b = perm[s:s + batch_size]
            opt.zero_grad()
            lossf(net([t[b] for t in ten]), y[b]).backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            v = lossf(net([t[va_i] for t in ten]), y[va_i]).item()
        if v < best - 1e-4:
            best, bad = v, 0
            best_state = {k: p.detach().clone() for k, p in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    net.load_state_dict(best_state)
    return net.eval(), va_i, ep

class NNFusion(FusionBase):
    """Shared training/tuning logic of M5 and M6 (native binary output through a BCE head)."""
    NATIVE_BINARY = True
    GRID = []

    def __init__(self, task_type="regression", blocks=None, random_state=0, grid=None, lr=2e-3, max_epochs=200,
                 patience=15, batch_size=128):
        super().__init__(task_type, blocks, random_state)
        self.grid, self.lr, self.max_epochs, self.patience, self.batch_size = grid, lr, max_epochs, patience, batch_size

    def _forward(self, net, mods):
        with torch.no_grad():
            out = net([torch.as_tensor(m, dtype=torch.float32) for m in mods])
        return torch.sigmoid(out).numpy() if self.task_type == "binary" else out.numpy()

    def _train_cfg(self, mods, target, groups, cfg, seed):
        torch.manual_seed(seed)
        net = self._new_net(cfg, mods)
        net, va, ep = train_net(net, mods, target, groups, self.task_type == "binary", self.lr,
                                cfg.get("weight_decay", 1e-2), self.max_epochs, self.patience, self.batch_size, seed)
        return net, va, ep

    def _fit_model(self, Z, ys, groups, y_raw, hyper=None):
        mods = self._modalities(Z, fit=True)
        binary = self.task_type == "binary"
        target = np.asarray(y_raw, float) if binary else ys
        grid = list(self.grid) if self.grid else list(self.GRID)
        if hyper is None:
            if len(grid) > 1:
                scores = []
                for ci, cfg in enumerate(grid):
                    sc = []
                    for a, b in inner_group_splits(groups):
                        net, _, _ = self._train_cfg([m[a] for m in mods], target[a], groups[a], cfg, self.random_state + ci)
                        pred = self._forward(net, [m[b] for m in mods])
                        sc.append(float(np.mean((pred - target[b]) ** 2)))      # Brier score / MSE on the held-out inner fold
                    scores.append(np.mean(sc))
                hyper = grid[int(np.argmin(scores))]
                self.inner_scores_ = {json.dumps(c, sort_keys=True): float(s) for c, s in zip(grid, scores)}
            else:
                hyper = grid[0]
        net, va, self.epochs_ = self._train_cfg(mods, target, groups, hyper, self.random_state)
        self.val_idx_ = va
        self.param_count_ = int(sum(p.numel() for p in net.parameters()))
        self.param_ratio_ = self.param_count_ / len(ys)
        return net, hyper

    def _predict_model(self, net, Z):
        return self._forward(net, self._modalities(Z, fit=False))


class M5LMF(NNFusion):
    """M5: low-rank multimodal fusion over [fingerprint, descriptors, cell, poi] (PCA 32 each) + meta (small blocks).
    Inner grid: rank in {2, 4, 8} x fusion_dim in {16, 32} (6 configs) x GroupKFold(3)."""
    REDUCE, PCA_K = True, 32
    GRID = [{"rank": r, "fusion_dim": h} for r in (2, 4, 8) for h in (16, 32)]

    def _modalities(self, Z, fit=False):
        meta = np.concatenate([Z[b] for b in SMALL_BLOCKS], axis=1)
        return [Z["fingerprint"], Z["descriptors"], Z["cell"], Z["poi"], meta]

    def _new_net(self, cfg, mods):
        return LMFNet([m.shape[1] for m in mods], cfg["rank"], cfg["fusion_dim"], dropout=0.1)


class M6FiLM(NNFusion):
    """M6: biological context (PCA(cell, poi) + small blocks) modulates the PCA-reduced molecular representation.
    Inner grid: hidden in {32, 64} x weight_decay in {1e-3, 1e-2} (4 configs) x GroupKFold(3)."""
    GRID = [{"hidden": hd, "weight_decay": wd} for hd in (32, 64) for wd in (1e-3, 1e-2)]

    def __init__(self, task_type="regression", blocks=None, random_state=0, grid=None, lr=2e-3, max_epochs=200,
                 patience=15, batch_size=128, ctx_dim=32):
        super().__init__(task_type, blocks, random_state, grid, lr, max_epochs, patience, batch_size)
        self.ctx_dim = ctx_dim

    def _modalities(self, Z, fit=False):
        mol = np.concatenate([Z["fingerprint"], Z["descriptors"]], axis=1)
        dense_ctx = np.concatenate([Z["cell"], Z["poi"]], axis=1)
        if fit:                                  # fitted on the training rows only (this is called from fit)
            k = lambda a: int(min(self.ctx_dim, a.shape[0] - 1, a.shape[1]))
            self.mol_pca_ = PCA(n_components=k(mol), random_state=self.random_state).fit(mol)
            self.ctx_pca_ = PCA(n_components=k(dense_ctx), random_state=self.random_state).fit(dense_ctx)
        small = np.concatenate([Z[b] for b in SMALL_BLOCKS], axis=1)
        return [self.mol_pca_.transform(mol), np.concatenate([self.ctx_pca_.transform(dense_ctx), small], axis=1)]

    def _new_net(self, cfg, mods):
        net = FiLMNet(mods[0].shape[1], mods[1].shape[1], cfg["hidden"], dropout=0.1)
        return _TwoInput(net)


class _TwoInput(nn.Module):
    """Adapter so that FiLMNet(mol, ctx) accepts the list-of-modalities interface used by ``train_net``."""
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, zs):
        return self.net(zs[0], zs[1])

In [ ]:
# TEST (written first): LMF (M5) and FiLM (M6) networks + estimators
def _test_nn():
    import time as _t
    from sklearn.metrics import r2_score, roc_auc_score
    # ---- LMF network: parameter formula and shapes
    dims = [32, 32, 32, 32, 24]
    for r, h in [(2, 16), (4, 32), (8, 32)]:
        net = LMFNet(dims, rank=r, fusion_dim=h)
        n_par = sum(p.numel() for p in net.parameters())
        assert n_par == lmf_param_count(dims, r, h) == sum(r * (d + 1) * h for d in dims) + h + 1
    zs = [torch.randn(7, d) for d in dims]
    assert LMFNet(dims, 4, 16)(zs).shape == (7,)
    # the constant 1 appended to each modality: with all inputs zero the output is still a function of the biases only
    z0 = [torch.zeros(3, d) for d in dims]
    out0 = LMFNet(dims, 4, 16)(z0); assert torch.allclose(out0, out0[0].expand(3))
    # ---- FiLM network starts as the identity modulation: output == head(h_mol)
    film = FiLMNet(d_mol=16, d_ctx=40, hidden=32, dropout=0.0)
    mol, ctx = torch.randn(5, 16), torch.randn(5, 40)
    assert torch.allclose(film(mol, ctx), film.head(mol).squeeze(-1)), "(1 + gamma) parameterisation must start at identity"
    assert sum(p.numel() for p in film.film.parameters()) == 40 * 32 + 32      # single linear layer -> (gamma, beta)
    # ---- estimators
    d = make_synthetic_data(n=1400, seed=8, interaction=(1.2, 0.9))
    X = assemble_X(d); g, _ = scaffold_groups(d["smiles"], verbose=False)
    y = d["y"]["dmax"]; ok = np.isfinite(y); X, y, g = X[ok], y[ok], g[ok]
    tr, te = make_splits(y, g, n_repeats=1, seeds=[6])[0][0]
    for cls in (M5LMF, M6FiLM):
        t0 = _t.time()
        est = cls(random_state=1, grid=cls.GRID[:2], max_epochs=60).fit(X[tr], y[tr], groups=g[tr])
        p = est.predict(X[te])
        assert p.shape == (len(te),) and np.isfinite(p).all()
        r2 = r2_score(y[te], p)
        assert r2 > 0.1, f"{cls.__name__} R2={r2:.3f}"
        assert est.hyper_ in cls.GRID[:2] or est.hyper_ == dict(est.hyper_)
        assert est.param_count_ > 0 and est.param_ratio_ == est.param_count_ / len(tr)
        assert not set(g[tr][est.val_idx_]) & set(g[tr][np.setdiff1d(np.arange(len(tr)), est.val_idx_)]), \
            "early-stopping validation split must be scaffold-grouped"
        p2 = cls(random_state=1, grid=cls.GRID[:2], max_epochs=60).fit(X[tr], y[tr], groups=g[tr]).predict(X[te])
        np.testing.assert_allclose(p2, p, atol=1e-5)                                 # seeded -> deterministic
        print(f"  {cls.__name__}: R2={r2:.3f} params={est.param_count_} ratio={est.param_ratio_:.1f} "
              f"hyper={est.hyper_} ({_t.time() - t0:.0f}s)")
    yb = d["y"]["activity"]; okb = np.isfinite(yb)
    Xb, yb, gb = assemble_X(d)[okb], yb[okb], scaffold_groups(d["smiles"][okb], verbose=False)[0]
    trb, teb = make_splits(yb, gb, n_repeats=1, seeds=[6])[0][0]
    for cls in (M5LMF, M6FiLM):
        pb = cls(task_type="binary", grid=cls.GRID[:1], max_epochs=40).fit(Xb[trb], yb[trb], groups=gb[trb]).predict(Xb[teb])
        assert ((pb >= 0) & (pb <= 1)).all() and roc_auc_score(yb[teb], pb) > 0.6, cls.__name__
    print("LMF / FiLM: OK")

_test_nn()

### 5.5 M7 (optional): XGBoost on the concatenation, PCA on the context only
Reference point only (set `FUSION_RUN_M7=0` to skip). The context blocks (`cell`, `poi`) are PCA-reduced to 32 components;
the Morgan bits and the RDKit descriptors are **not** reduced (trees split on individual bits and descriptors, which a PCA
rotation would smear over all components) and are only mean-imputed; small blocks passed through; shallow trees, strong L2 and `min_child_weight`, row/column subsampling, early stopping on a scaffold-grouped
validation split. Inner grid: `max_depth in {3, 5}` x `reg_lambda in {5, 20}` x `GroupKFold(3)`.

In [ ]:
class M7XGB(FusionBase):
    """M7: regularised gradient-boosted trees on [raw fingerprint, raw descriptors, PCA(32) of cell and poi, small blocks].

    PCA is applied to the biological context only: trees split on individual Morgan bits and RDKit descriptors, which a
    PCA rotation would smear over all columns. The molecule blocks are only mean-imputed (trees are scale-free).
    """
    NATIVE_BINARY = True
    REDUCE, PCA_K = True, 32
    REDUCE_BLOCKS = ("cell", "poi")            # the context blocks; fingerprint and descriptors are not reduced
    GRID = [{"max_depth": d, "reg_lambda": lam} for d in (3, 5) for lam in (5.0, 20.0)]

    def __init__(self, task_type="regression", blocks=None, random_state=0, grid=None, n_estimators=400,
                 learning_rate=0.05, n_jobs=1):
        super().__init__(task_type, blocks, random_state)
        self.grid, self.n_estimators, self.learning_rate, self.n_jobs = grid, n_estimators, learning_rate, n_jobs

    def _make_preprocessor(self):
        return BlockPreprocessor(blocks=self.blocks, reduce=True, pca_k=self.PCA_K, random_state=self.random_state,
                                 reduce_blocks=self.REDUCE_BLOCKS, scale_blocks=self.REDUCE_BLOCKS)

    def _new(self, cfg):
        kw = dict(n_estimators=self.n_estimators, learning_rate=self.learning_rate, max_depth=cfg["max_depth"],
                  reg_lambda=cfg["reg_lambda"], min_child_weight=5, subsample=0.8, colsample_bytree=0.5, gamma=0.1,
                  tree_method="hist", n_jobs=self.n_jobs, random_state=self.random_state, early_stopping_rounds=30,
                  verbosity=0)
        return xgb.XGBClassifier(**kw) if self.task_type == "binary" else xgb.XGBRegressor(**kw)

    def _fit_cfg(self, A, target, groups, cfg):
        split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=self.random_state)
        tr, va = next(split.split(A, groups=groups))
        m = self._new(cfg)
        m.fit(A[tr], target[tr], eval_set=[(A[va], target[va])], verbose=False)
        return m

    def _score(self, m, A):
        return m.predict_proba(A)[:, 1] if self.task_type == "binary" else m.predict(A)

    def _fit_model(self, Z, ys, groups, y_raw, hyper=None):
        A = self.pre_.concat(Z)
        target = np.asarray(y_raw, float) if self.task_type == "binary" else ys
        grid = list(self.grid) if self.grid else list(self.GRID)
        if hyper is None:
            if len(grid) > 1:
                scores = []
                for cfg in grid:
                    sc = [np.mean((self._score(self._fit_cfg(A[a], target[a], groups[a], cfg), A[b]) - target[b]) ** 2)
                          for a, b in inner_group_splits(groups)]
                    scores.append(np.mean(sc))
                hyper = grid[int(np.argmin(scores))]
            else:
                hyper = grid[0]
        m = self._fit_cfg(A, target, groups, hyper)
        self.n_trees_ = int(m.best_iteration) + 1
        return m, hyper

    def _predict_model(self, model, Z):
        return self._score(model, self.pre_.concat(Z))

In [ ]:
# TEST (written first): optional XGBoost reference (M7)
def _test_m7():
    from sklearn.metrics import r2_score, roc_auc_score
    X, y, g, tr, te = _synthetic_fold()
    est = M7XGB(random_state=0, grid=M7XGB.GRID[:2]).fit(X[tr], y[tr], groups=g[tr])
    p = est.predict(X[te])
    assert p.shape == (len(te),) and r2_score(y[te], p) > 0.1, r2_score(y[te], p)
    assert est.hyper_ in M7XGB.GRID[:2] and 1 <= est.n_trees_ <= 400
    np.testing.assert_allclose(M7XGB(random_state=0, grid=M7XGB.GRID[:2]).fit(X[tr], y[tr], groups=g[tr]).predict(X[te]), p, atol=1e-6)
    # PCA only on the context: the molecule blocks reach the trees as raw columns, cell/poi as 32 components
    dims = est.pre_.dims_
    assert dims["fingerprint"] == len(BLOCKS["fingerprint"]) and dims["descriptors"] == len(BLOCKS["descriptors"]), dims
    assert dims["cell"] == dims["poi"] == M7XGB.PCA_K, dims
    assert est.model_.n_features_in_ == sum(dims.values())
    Zt = est.pre_.transform(X[te])
    np.testing.assert_array_equal(Zt["fingerprint"], np.nan_to_num(X[te][:, BLOCKS["fingerprint"]]))
    Xb, yb, gb, trb, teb = _synthetic_fold(task="activity")
    pb = M7XGB(task_type="binary", grid=M7XGB.GRID[:1]).fit(Xb[trb], yb[trb], groups=gb[trb]).predict(Xb[teb])
    assert ((pb >= 0) & (pb <= 1)).all() and roc_auc_score(yb[teb], pb) > 0.6
    print(f"M7 XGBoost: OK (R2={r2_score(y[te], p):.3f})")

_test_m7()

### 5.6 Method registry
`make_methods(fast=False)` maps method names to factories `(task_type, seed) -> estimator`, so the harness treats all
methods uniformly. `fast=True` (smoke tests only) shortens the optimisers and truncates the inner grids.

| method | model | inner tuning (scaffold-grouped `GroupKFold(3)` on the training fold) |
|---|---|---|
| M1_ridge | ridge on scaled concatenation | `RidgeCV`, 15 alphas 1e-2..1e5 |
| M2_stack | 5 ridge experts + non-negative stacking | per-expert alpha grid as M1; OOF by `GroupKFold(5)` |
| M3_gp_add | additive-kernel GP | marginal likelihood, 3 restarts (no inner CV) |
| M4_gp_inter | + product kernels | marginal likelihood, 3 restarts (no inner CV) |
| M5_lmf | low-rank multimodal fusion | rank {2,4,8} x fusion dim {16,32} |
| M6_film | FiLM | hidden {32,64} x weight decay {1e-3,1e-2} |
| M7_xgb (optional) | XGBoost on raw molecule blocks + PCA(cell, poi) + small blocks | depth {3,5} x lambda {5,20} |

In [ ]:
def make_methods(fast=False, include_m7=None):
    """Dict method name -> factory ``(task_type, seed) -> estimator`` for M1-M6 (+ M7 if enabled)."""
    include_m7 = RUN_M7 if include_m7 is None else include_m7
    gp_kw = dict(n_restarts=2, n_iter=20, max_hyper_points=400) if fast else dict(n_restarts=3, n_iter=GP_N_ITER, max_hyper_points=GP_HYPER_POINTS)
    nn_kw = dict(max_epochs=40, patience=8) if fast else dict(max_epochs=NN_MAX_EPOCHS, patience=NN_PATIENCE)
    cut = (lambda grid: grid[:2]) if fast else (lambda grid: grid)
    methods = {
        "M1_ridge": lambda tt, seed: M1Ridge(task_type=tt, random_state=seed),
        "M2_stack": lambda tt, seed: M2Stack(task_type=tt, random_state=seed),
        "M3_gp_add": lambda tt, seed: GPFusion(task_type=tt, random_state=seed, interactions=False, **gp_kw),
        "M4_gp_inter": lambda tt, seed: GPFusion(task_type=tt, random_state=seed, interactions=True, **gp_kw),
        "M5_lmf": lambda tt, seed: M5LMF(task_type=tt, random_state=seed, grid=cut(M5LMF.GRID), **nn_kw),
        "M6_film": lambda tt, seed: M6FiLM(task_type=tt, random_state=seed, grid=cut(M6FiLM.GRID), **nn_kw),
    }
    if include_m7:
        n_est = 100 if fast else 400
        methods["M7_xgb"] = lambda tt, seed: M7XGB(task_type=tt, random_state=seed, grid=cut(M7XGB.GRID), n_estimators=n_est)
    return methods

### 5.7 Experiment harness
For every (task, method, repeat, fold) one row is appended to `results.csv`: R², RMSE, MAE, Spearman, ROC-AUC
(thresholded for Dmax / pDC50, native for activity), log loss and Brier score (activity), fit time, `n_train`, `n_test`,
the hyper-parameters chosen by the inner CV and method diagnostics (M2 weights, GP kernel weights, parameter counts).
* **Resumable**: cells already in `results.csv` are skipped (set `RETRY_FAILED = True` to also retry failed cells).
* **Failure-proof**: an exception becomes a NaN row (`status = "error"`, traceback in `error`).
* Fold predictions are pickled per cell in `predictions/` and consolidated into `predictions.pkl`.
* Estimators receive **only training rows** in `fit` and **only test rows** in `predict`; the processing state
  (scalers, PCA, imputers, target standardisation) lives inside the estimator and is fitted there.
* `N_JOBS > 1` (env `FUSION_N_JOBS`) runs folds in parallel worker processes; each is single-threaded for OpenMP
  (see the setup cell: torch and xgboost cannot share a multi-threaded OpenMP runtime on macOS).

In [ ]:
import sys, types, glob
from scipy.stats import spearmanr
from sklearn.metrics import r2_score, roc_auc_score, log_loss, mean_squared_error, mean_absolute_error

RESULT_COLUMNS = ["task", "method", "repeat", "fold", "fold_id", "r2", "rmse", "mae", "spearman", "roc_auc", "log_loss",
                  "brier", "fit_time", "n_train", "n_test", "hyper", "diag", "status", "error"]
RETRY_FAILED = False

def score_fold(task, y_true, y_pred):
    """All fold metrics; ROC-AUC is NaN if the test fold has a single class, log loss/Brier only for ``activity``."""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    out = {"r2": r2_score(y_true, y_pred), "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
           "mae": float(mean_absolute_error(y_true, y_pred)), "log_loss": np.nan, "brier": np.nan}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        out["spearman"] = float(spearmanr(y_true, y_pred)[0])
    labels = (y_true > AUC_THRESHOLD[task]).astype(int) if task in AUC_THRESHOLD else y_true.astype(int)
    out["roc_auc"] = float(roc_auc_score(labels, y_pred)) if len(np.unique(labels)) == 2 else np.nan
    if task == "activity":
        p = np.clip(y_pred, 1e-6, 1 - 1e-6)
        out["log_loss"] = float(log_loss(labels, p, labels=[0, 1]))
        out["brier"] = float(np.mean((labels - y_pred) ** 2))
    return out

def set_worker_state(tasks):
    """Make the task views visible to ``run_fold`` (in this process or, via the initializer, in a worker)."""
    mod = sys.modules.get("fusion_state") or types.ModuleType("fusion_state")
    mod.TASKS = tasks
    sys.modules["fusion_state"] = mod

def _worker_init(tasks):
    """Worker start-up: threading is inherited from the environment (single-threaded OpenMP, see setup)."""
    import xgboost                      # noqa: F401
    import torch                        # noqa: F401
    warnings.filterwarnings("ignore", message=".*added jitter.*")        # GPyTorch Cholesky jitter notices
    warnings.filterwarnings("ignore", category=FutureWarning)
    set_worker_state(tasks)

def _json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items() if not hasattr(v, "state_dict") and k != "state"}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return [float(v) for v in obj]
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    return obj

def collect_diag(est):
    """Method-specific diagnostics worth keeping per fold (M2 weights, GP kernel weights, NN size, trees)."""
    d = {}
    if hasattr(est, "weights_") and hasattr(est, "expert_names_"):
        d.update(experts=est.expert_names_, weights=est.weights_, weight_shares=est.weight_shares_)
    if hasattr(est, "kernel_weights_"):
        d.update(kernel_weights=est.kernel_weights_, kernel_report=est.kernel_report_)
    if hasattr(est, "param_count_"):
        d.update(param_count=est.param_count_, param_ratio=est.param_ratio_, flag_ratio_gt_10=bool(est.param_ratio_ > 10),
                 epochs=getattr(est, "epochs_", None), inner_scores=getattr(est, "inner_scores_", None))
    if hasattr(est, "n_trees_"):
        d["n_trees"] = est.n_trees_
    return _json_safe(d)

def run_fold(task, method, repeat, fold, factory):
    """Fit one method on the training rows of one fold and score it on the test rows.

    Returns:
        (row, preds): the results row (dict, NaN metrics + traceback on failure) and the fold predictions (None on failure).
    """
    view = sys.modules["fusion_state"].TASKS[task]
    tr, te = view["splits"][repeat][fold]
    seed = GLOBAL_SEED + 100 * repeat + fold
    row = {c: np.nan for c in RESULT_COLUMNS}
    row.update(task=task, method=method, repeat=repeat, fold=fold, fold_id=repeat * N_SPLITS + fold,
               n_train=len(tr), n_test=len(te), status="ok", error="", hyper="{}", diag="{}")
    try:
        set_all_seeds(seed)
        est = factory("binary" if task == "activity" else "regression", seed)
        t0 = time.time()
        est.fit(view["X"][tr], view["y"][tr], groups=view["groups"][tr])
        row["fit_time"] = time.time() - t0
        pred = np.asarray(est.predict(view["X"][te]), dtype=float)
        if pred.shape != (len(te),) or not np.isfinite(pred).all():
            raise ValueError(f"predictions must be finite with shape ({len(te)},), got {pred.shape}")
        row.update(score_fold(task, view["y"][te], pred))
        row["hyper"] = json.dumps(_json_safe(getattr(est, "hyper_", {})), sort_keys=True)
        row["diag"] = json.dumps(collect_diag(est), sort_keys=True)
        return row, {"test_idx": te, "y_true": view["y"][te].copy(), "y_pred": pred}
    except Exception:
        row.update(status="error", error=traceback.format_exc())
        return row, None

def _results_path(d):
    return Path(d) / "results.csv"

def _append_row(d, row):
    path = _results_path(d)
    pd.DataFrame([row], columns=RESULT_COLUMNS).to_csv(path, mode="a", header=not path.exists(), index=False)

def _save_predictions(d, key, preds):
    (Path(d) / "predictions").mkdir(parents=True, exist_ok=True)
    with open(Path(d) / "predictions" / ("__".join(map(str, key)) + ".pkl"), "wb") as f:
        pickle.dump(preds, f)

def load_predictions(d):
    """Dict (task, method, repeat, fold) -> {'test_idx', 'y_true', 'y_pred'} from the per-cell pickles."""
    out = {}
    for f in sorted(glob.glob(str(Path(d) / "predictions" / "*.pkl"))):
        task, method, repeat, fold = Path(f).stem.split("__")
        with open(f, "rb") as fh:
            out[(task, method, int(repeat), int(fold))] = pickle.load(fh)
    return out

def _check_manifest(d, tasks, task_data):
    """Refuse to append to a results directory that was produced from different data/splits."""
    path = Path(d) / "run_manifest.json"
    man = json.loads(path.read_text()) if path.exists() else {"tasks": {}}
    for t in tasks:
        v = task_data[t]
        sig = hashlib.sha1(np.ascontiguousarray(v["y"]).tobytes() + np.ascontiguousarray(v["groups"]).tobytes()
                           + np.ascontiguousarray(v["idx"]).tobytes()).hexdigest()
        if man["tasks"].get(t, sig) != sig:
            raise RuntimeError(f"results directory {d} holds results for different data/splits of task '{t}'. "
                               f"Use another RESULTS_DIR or delete it.")
        man["tasks"][t] = sig
    man["config"] = {"n_splits": N_SPLITS, "seeds": REPEAT_SEEDS, "strat_bins": N_STRAT_BINS, "dmax_thr": DMAX_THR,
                     "pdc50_thr": PDC50_THR, "inner_folds": INNER_FOLDS}
    path.write_text(json.dumps(man, indent=1))

def run_experiments(tasks, methods, n_repeats, n_folds, results_dir, n_jobs=1, progress=True, task_data=None):
    """Run every (task, method, repeat, fold) cell that is not yet in ``results.csv``.

    Args:
        tasks: task names to run.
        methods: dict method name -> factory ``(task_type, seed) -> estimator``.
        n_repeats, n_folds: repeats and folds per repeat to run.
        results_dir: directory of ``results.csv``, ``predictions/`` and ``run_manifest.json``.
        n_jobs: parallel worker processes (1 = in-process).
        progress: show a tqdm progress bar and print the wall-clock estimate after the first (task, method) block.
        task_data: task views (default: the ones registered with ``set_worker_state``).
    """
    results_dir = Path(results_dir); results_dir.mkdir(parents=True, exist_ok=True)
    if task_data is not None:
        set_worker_state(task_data)
    task_data = sys.modules["fusion_state"].TASKS
    _check_manifest(results_dir, tasks, task_data)
    path = _results_path(results_dir)
    done = set()
    if path.exists():
        prev = pd.read_csv(path)
        if RETRY_FAILED and (prev["status"] == "error").any():
            prev = prev[prev["status"] == "ok"]; prev.to_csv(path, index=False)
        done = set(zip(prev["task"], prev["method"], prev["repeat"], prev["fold"]))
    jobs = [(t, m, r, f) for t in tasks for m in methods for r in range(n_repeats) for f in range(n_folds)
            if (t, m, r, f) not in done]
    if not jobs:
        print("nothing to do: every requested (task, method, repeat, fold) cell is already in results.csv")
        return
    block_total = {}
    for t, m, _, _ in jobs:
        block_total[(t, m)] = block_total.get((t, m), 0) + 1
    first_block, block_done, t_start, reported = (jobs[0][0], jobs[0][1]), {}, time.time(), False
    bar = tqdm(total=len(jobs), desc="folds", disable=not progress, mininterval=30, maxinterval=120)

    def _finish(job, row, preds):
        nonlocal reported
        _append_row(results_dir, row)
        if preds is not None:
            _save_predictions(results_dir, job, preds)
        bar.update(1); bar.set_postfix_str(f"{job[0]}/{job[1]}"); block_done[job[:2]] = block_done.get(job[:2], 0) + 1
        if progress and not reported and block_done.get(first_block, 0) == block_total[first_block]:
            reported = True
            dt = time.time() - t_start
            est_total = dt * len(jobs) / block_total[first_block]
            tqdm.write(f"first block {first_block} ({block_total[first_block]} fits, n_jobs={n_jobs}) took {dt / 60:.1f} min -> "
                       f"naive estimate for all {len(jobs)} pending fits: {est_total / 60:.0f} min "
                       f"(methods differ in cost: GPs and neural nets are slower than ridge)")

    if n_jobs <= 1:
        for job in jobs:
            row, preds = run_fold(*job, methods[job[1]])
            _finish(job, row, preds)
    else:
        from joblib.externals.loky import ProcessPoolExecutor, as_completed
        with ProcessPoolExecutor(max_workers=n_jobs, initializer=_worker_init, initargs=(task_data,), timeout=3600) as ex:
            futs = {ex.submit(run_fold, *job, methods[job[1]]): job for job in jobs}
            for fut in as_completed(futs):
                row, preds = fut.result()
                _finish(futs[fut], row, preds)
    bar.close()
    with open(results_dir / "predictions.pkl", "wb") as f:
        pickle.dump(load_predictions(results_dir), f)

In [ ]:
# TEST (written first): fold metrics
def _test_metrics():
    from sklearn.metrics import log_loss
    y = np.array([0.1, 0.5, 0.9, 0.85, 0.3, 0.95, 0.7, 0.2])
    m = score_fold("dmax", y, y)
    assert m["r2"] == 1 and m["rmse"] == 0 and m["mae"] == 0 and m["spearman"] == 1 and m["roc_auc"] == 1
    m = score_fold("dmax", y, -y)                      # Dmax > 0.8 are the positives (0.9, 0.85, 0.95)
    assert m["spearman"] == -1 and m["roc_auc"] == 0
    assert abs(score_fold("dmax", y, np.full_like(y, y.mean()))["r2"]) < 1e-12
    # pDC50: positives are pDC50 > 6 (strictly)
    yp = np.array([4.0, 6.0, 6.5, 7.0, 8.0, 5.5])
    assert score_fold("pdc50", yp, yp + 0.01 * np.arange(6))["roc_auc"] == 1.0
    assert score_fold("pdc50", yp, -yp)["roc_auc"] == 0.0
    # activity: ROC-AUC and log loss on probabilities
    ya, pa = np.array([1.0, 0.0, 1.0, 0.0]), np.array([0.9, 0.2, 0.6, 0.4])
    m = score_fold("activity", ya, pa)
    assert m["roc_auc"] == 1.0 and abs(m["log_loss"] - log_loss(ya, pa)) < 1e-12
    assert abs(m["brier"] - np.mean((ya - pa) ** 2)) < 1e-12
    assert np.isfinite(score_fold("activity", ya, np.array([1.0, 0.0, 1.0, 1.0]))["log_loss"]), "probabilities are clipped"
    # a test fold with a single class has no AUC (NaN) but must not raise
    assert np.isnan(score_fold("dmax", np.array([0.1, 0.2, 0.3]), np.array([0.1, 0.2, 0.3]))["roc_auc"])
    assert PRIMARY_METRIC == {"dmax": "r2", "pdc50": "r2", "activity": "roc_auc"}
    print("score_fold: OK")

_test_metrics()

In [ ]:
# TEST (written first): fold runner protocol, error rows, resumable results.csv
class _SpyEstimator(BaseEstimator):
    """Records what the harness hands to fit / predict, and predicts the training mean."""
    calls = []
    def __init__(self, task_type="regression", fail=False):
        self.task_type, self.fail = task_type, fail
    def fit(self, X, y, groups=None):
        if self.fail:
            raise ValueError("boom")
        _SpyEstimator.calls.append(("fit", X.copy(), np.asarray(y).copy(), None if groups is None else np.asarray(groups).copy()))
        self.mu_ = float(np.mean(y)); self.hyper_ = {"mu": self.mu_}
        return self
    def predict(self, X):
        _SpyEstimator.calls.append(("predict", X.copy()))
        return np.full(len(X), self.mu_)

def _test_harness():
    import shutil
    d = make_synthetic_data(n=500, seed=4)
    X = assemble_X(d); g, _ = scaffold_groups(d["smiles"], verbose=False)
    out = RESULTS_DIR / "unittest_harness"; shutil.rmtree(out, ignore_errors=True)
    tasks = build_tasks(d, X, g, ["dmax", "activity"], n_repeats=2, results_dir=out)
    set_worker_state(tasks)
    # (1) the estimator only ever sees training rows in fit and test rows in predict
    _SpyEstimator.calls.clear()
    v = tasks["dmax"]; tr, te = v["splits"][1][2]
    row, preds = run_fold("dmax", "SPY", 1, 2, lambda tt, seed: _SpyEstimator(tt))
    fit_call, pred_call = _SpyEstimator.calls
    np.testing.assert_array_equal(fit_call[1], v["X"][tr]); np.testing.assert_array_equal(fit_call[2], v["y"][tr])
    np.testing.assert_array_equal(fit_call[3], v["groups"][tr]); np.testing.assert_array_equal(pred_call[1], v["X"][te])
    assert not set(v["groups"][tr]) & set(v["groups"][te])
    for k in ["task", "method", "repeat", "fold", "fold_id", "r2", "rmse", "mae", "spearman", "roc_auc", "log_loss",
              "fit_time", "n_train", "n_test", "hyper", "status", "error"]:
        assert k in row, k
    assert row["status"] == "ok" and row["fold_id"] == 1 * N_SPLITS + 2 and row["n_train"] == len(tr) and row["n_test"] == len(te)
    assert json.loads(row["hyper"]) == {"mu": float(np.mean(v["y"][tr]))}
    np.testing.assert_array_equal(preds["test_idx"], te); np.testing.assert_array_equal(preds["y_true"], v["y"][te])
    # (2) a failing method yields a NaN row with the traceback instead of raising
    row, preds = run_fold("dmax", "BAD", 0, 0, lambda tt, seed: _SpyEstimator(tt, fail=True))
    assert row["status"] == "error" and "boom" in row["error"] and np.isnan(row["r2"]) and preds is None
    # (3) resumable harness: finished (method, repeat, fold) cells are skipped, a deleted cell is refitted
    facs = {"SPYA": lambda tt, seed: _SpyEstimator(tt), "SPYB": lambda tt, seed: _SpyEstimator(tt)}
    kw = dict(tasks=["dmax", "activity"], methods=facs, n_repeats=1, n_folds=2, results_dir=out, n_jobs=1, progress=False)
    _SpyEstimator.calls.clear()
    run_experiments(**kw)
    res = pd.read_csv(out / "results.csv")
    assert len(res) == 2 * 2 * 1 * 2 and not res.duplicated(["task", "method", "repeat", "fold"]).any()
    n_fits = sum(c[0] == "fit" for c in _SpyEstimator.calls); assert n_fits == 8
    _SpyEstimator.calls.clear(); run_experiments(**kw)
    assert not _SpyEstimator.calls, "second run must skip every finished cell"
    res2 = res[~((res.task == "activity") & (res.method == "SPYB") & (res.fold == 1))]
    res2.to_csv(out / "results.csv", index=False)
    run_experiments(**kw)
    assert sum(c[0] == "fit" for c in _SpyEstimator.calls) == 1 and len(pd.read_csv(out / "results.csv")) == 8
    # (4) NaN rows do not kill the run and are recorded once
    bad = {"BAD": lambda tt, seed: _SpyEstimator(tt, fail=True)}
    run_experiments(**{**kw, "methods": bad, "tasks": ["dmax"]})
    res3 = pd.read_csv(out / "results.csv"); b = res3[res3.method == "BAD"]
    assert len(b) == 2 and (b.status == "error").all() and b.error.str.contains("boom").all()
    # (5) predictions are pickled and can be reloaded without refitting
    P = load_predictions(out)
    assert ("dmax", "SPYA", 0, 0) in P and len(P[("dmax", "SPYA", 0, 0)]["y_pred"]) == len(tasks["dmax"]["splits"][0][0][1])
    # (6) data changed under the same results directory -> refuse to mix old and new results
    tasks["dmax"]["y"] = tasks["dmax"]["y"] + 1.0
    try:
        run_experiments(**kw)
    except RuntimeError as e:
        assert "different" in str(e)
    else:
        raise AssertionError("stale results directory must be rejected")
    shutil.rmtree(out, ignore_errors=True)
    print("harness: OK")

_test_harness()

## 6. Smoke test on synthetic data
The whole pipeline (scaffold groups, splits, all methods, parallel harness, results/predictions files) on
`make_synthetic_data` with **1 repeat x 2 folds** and shortened optimisers (`fast=True`). It re-registers the real
data afterwards. On this data M4 (and the FiLM/LMF models) should beat the purely additive M3: the planted
signal contains genuine molecule x protein and cell x protein interactions.

In [ ]:
def run_smoke_test(n_jobs=2):
    import shutil
    smoke_dir = RESULTS_DIR / "smoke"; shutil.rmtree(smoke_dir, ignore_errors=True)
    saved_blocks = {k: v.copy() for k, v in BLOCKS.items()}
    try:
        sd = make_synthetic_data(n=1200, seed=0)
        Xs = assemble_X(sd)
        gs, _ = scaffold_groups(sd["smiles"], verbose=False)
        tasks_s = build_tasks(sd, Xs, gs, ALL_TASKS, n_repeats=1, results_dir=smoke_dir)
        for v in tasks_s.values():
            assert_no_group_leakage(v)
        methods = make_methods(fast=True, include_m7=True)
        t0 = time.time()
        run_experiments(ALL_TASKS, methods, n_repeats=1, n_folds=2, results_dir=smoke_dir, n_jobs=n_jobs, task_data=tasks_s)
        res = pd.read_csv(smoke_dir / "results.csv")
        errs = res[res.status != "ok"]
        assert errs.empty, f"smoke-test failures:\n{errs[['task', 'method', 'error']].to_string()}"
        assert len(res) == len(ALL_TASKS) * len(methods) * 1 * 2 and set(res.method) == set(methods)
        assert len(load_predictions(smoke_dir)) == len(res) and (smoke_dir / "predictions.pkl").exists()
        pm = res.pivot_table(index="method", columns="task", values=[PRIMARY_METRIC[t] for t in ALL_TASKS][0], aggfunc="mean")
        summ = pd.DataFrame({t: res[res.task == t].groupby("method")[PRIMARY_METRIC[t]].mean() for t in ALL_TASKS})
        print(f"smoke test: {len(res)} fits in {time.time() - t0:.0f}s  (primary metric: R2 for dmax/pdc50, ROC-AUC for activity)")
        print(summ.round(3))
        assert summ.loc["M4_gp_inter", "dmax"] > summ.loc["M3_gp_add", "dmax"], "M4 should find the planted interaction"
        assert (summ.loc["M1_ridge"] > summ.loc["M1_ridge"].min() - 1).all()
        w = res[(res.method == "M2_stack")].iloc[0]["diag"]
        assert "weights" in json.loads(w)
        return res
    finally:
        BLOCKS.clear(); BLOCKS.update(saved_blocks)
        if "TASK_DATA" in globals():
            set_worker_state(TASK_DATA)

smoke_results = run_smoke_test()

## 7. Full run
`FAST_MODE=1` runs 1 repeat x 3 folds (separate `fusion_results_fast/` directory); the default is 5 x 5 = 25 fits per
method and task. The loop is **resumable** (rerun this cell after an interruption) and prints a progress bar and, after
the first (task, method) block, a wall-clock estimate. `FUSION_N_JOBS` sets the number of worker processes.
Heavy cell: the complete run takes about an hour with 4 workers on a laptop.

Before running: the **LMF parameter budget** per task. The ratio parameters / training points should stay below
roughly 10; larger configurations are flagged (and flagged again per fit in `results.csv`).

In [ ]:
# the smoke test replaced the global state: re-register the real data
X_all = assemble_X(data)
set_worker_state(TASK_DATA)
METHODS = make_methods(fast=False)

n_meta = int(sum(len(BLOCKS[b]) for b in SMALL_BLOCKS))
lmf_dims = [M5LMF.PCA_K] * len(DENSE_BLOCKS) + [n_meta]
budget = []
for t, v in TASK_DATA.items():
    n_train = int(len(v["y"]) * (N_SPLITS - 1) / N_SPLITS)
    for cfg in M5LMF.GRID:
        p = lmf_param_count(lmf_dims, cfg["rank"], cfg["fusion_dim"])
        budget.append({"task": t, "rank": cfg["rank"], "fusion_dim": cfg["fusion_dim"], "params": p,
                       "n_train": n_train, "params/train_point": p / n_train, "FLAG (>10)": p / n_train > 10})
budget = pd.DataFrame(budget)
print("M5 LMF parameter budget (modalities: " + ", ".join(f"{b}:{M5LMF.PCA_K}" for b in DENSE_BLOCKS) + f", meta:{n_meta}):")
display(budget.round(2))
if budget["FLAG (>10)"].any():
    print("FLAG: some LMF configurations have more than ~10 parameters per training point; the inner CV and early stopping "
          "must keep them in check, and their selection frequency is reported below.")

print(f"\nmethods: {list(METHODS)}   tasks: {TASKS}   folds: {N_REPEATS} repeats x {N_FOLDS} folds   workers: {N_JOBS}")
t_run = time.time()
run_experiments(TASKS, METHODS, N_REPEATS, N_FOLDS, RESULTS_DIR, n_jobs=N_JOBS)
print(f"\nrun finished in {(time.time() - t_run) / 60:.1f} min (this session)")

In [ ]:
results = pd.read_csv(RESULTS_DIR / "results.csv")
print(f"results.csv: {len(results)} rows -> {RESULTS_DIR / 'results.csv'}")
cnt = results.pivot_table(index="method", columns="task", values="fold_id", aggfunc="count", fill_value=0)
display(cnt)
failed = results[results["status"] != "ok"]
print(f"\nfailed cells (NaN rows with traceback): {len(failed)}")
if len(failed):
    display(failed[["task", "method", "repeat", "fold", "error"]].assign(error=lambda d: d["error"].str.strip().str.split("\n").str[-1]))
display(results[results.status == "ok"].pivot_table(index="method", columns="task", values="fit_time", aggfunc="mean").round(1)
        .rename_axis(columns="mean fit time [s]"))
lmf = results[(results.method == "M5_lmf") & (results.status == "ok")]
if len(lmf):
    dg = pd.DataFrame([json.loads(d) for d in lmf["diag"]]).assign(task=lmf["task"].values, cfg=lmf["hyper"].values)
    print("\nM5 LMF: configurations chosen by the inner CV and parameters per training point")
    display(dg.groupby(["task", "cfg"]).agg(fits=("param_count", "size"), params=("param_count", "mean"),
                                             ratio=("param_ratio", "mean"), flagged=("flag_ratio_gt_10", "sum")).round(2))

## 8. Statistics
### 8.0 Definitions
All tests run **within a task** on the fold-level scores of the task's primary metric (R² for Dmax and pDC50,
ROC-AUC for activity), one value per (repeat, fold), 25 per method. Only *complete* folds enter the tests: if a method
failed on a fold (NaN row), that fold is dropped for all methods so the design stays balanced.
The folds are the repeated-measures "subjects" (`fold_id = repeat * 5 + fold`): every method is scored on exactly the
same train/test splits, so fold difficulty is a shared nuisance factor that paired analyses remove.

In [ ]:
import pingouin as pg
from scipy import stats as sps
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests

def wide_scores(res, task, metric, methods=None):
    """Complete-case wide table (rows = fold_id, columns = methods) of ``metric`` for ``task`` and #folds dropped."""
    d = res[res["task"] == task]
    if methods is not None:
        d = d[d["method"].isin(methods)]
    d = d.assign(value=np.where(d["status"] == "ok", d[metric], np.nan))
    wide = d.pivot(index="fold_id", columns="method", values="value").sort_index(axis=1)
    complete = wide.dropna(axis=0, how="any")
    return complete, int(d["fold_id"].nunique() - len(complete))

def descriptive_table(wide):
    q = wide.quantile([0.25, 0.75])
    out = pd.DataFrame({"mean": wide.mean(), "sd": wide.std(ddof=1), "median": wide.median(),
                        "iqr": q.loc[0.75] - q.loc[0.25], "min": wide.min(), "max": wide.max()})
    return out.sort_values("mean", ascending=False)

def _to_long(wide, metric):
    return wide.reset_index().melt(id_vars="fold_id", var_name="method", value_name=metric)

def rm_residuals(wide):
    """Residuals of the additive subject + method model (the repeated-measures ANOVA residuals)."""
    return wide - wide.mean(axis=1).to_numpy()[:, None] - wide.mean(axis=0).to_numpy()[None, :] + wide.to_numpy().mean()

def assumption_checks(wide, alpha=0.05, bad_alpha=0.01):
    """Levene (median-centred) across methods, Shapiro-Wilk per method on the ANOVA residuals, plain-language verdict."""
    lev = sps.levene(*[wide[c] for c in wide], center="median")
    resid = rm_residuals(wide)
    shapiro = {c: sps.shapiro(resid[c]) for c in wide}
    n_rej = sum(p < alpha for _, p in shapiro.values())
    n_bad = sum(p < bad_alpha for _, p in shapiro.values())
    equal_var = lev.pvalue >= alpha
    min_p = min(p for _, p in shapiro.values())
    bad = bool(lev.pvalue < bad_alpha or min_p < 1e-3 or n_bad >= int(np.ceil(len(shapiro) / 2)))
    verdict = (f"Homogeneity of variance {'holds' if equal_var else 'is violated'} (Levene, median-centred, p = {lev.pvalue:.3g}). "
               f"Residual normality {'holds' if n_rej == 0 else 'is violated'} for {len(shapiro) - n_rej} of {len(shapiro)} methods "
               f"(Shapiro-Wilk, alpha = {alpha}; smallest p = {min(p for _, p in shapiro.values()):.3g}). "
               + ("The assumptions are BADLY violated (rule: Levene p < 0.01, or any Shapiro p < 0.001, or at least half the methods "
                  "with Shapiro p < 0.01) -> the non-parametric tests (Friedman, Wilcoxon-Holm) are the ones to report."
                  if bad else "Neither assumption is badly violated (rule: Levene p < 0.01, or any Shapiro p < 0.001, or at least half "
                              "the methods with Shapiro p < 0.01), so the parametric analysis is reported."))
    return {"levene_stat": float(lev.statistic), "levene_p": float(lev.pvalue),
            "shapiro": {c: (float(w), float(p)) for c, (w, p) in shapiro.items()},
            "equal_var": bool(equal_var), "n_normal_rejected": int(n_rej), "badly_violated": bad, "verdict": verdict,
            "residuals": resid}

def rm_anova_report(wide, metric):
    """Repeated-measures ANOVA (folds = subjects) with Mauchly's sphericity test, Greenhouse-Geisser correction when
    sphericity is violated, and partial eta squared."""
    long = _to_long(wide, metric)
    table = pg.rm_anova(data=long, dv=metric, within="method", subject="fold_id", detailed=True, effsize="np2")
    eff = table.iloc[0]
    sph = pg.sphericity(long, dv=metric, within="method", subject="fold_id", method="mauchly") if wide.shape[1] > 2 else None
    spher_ok = True if sph is None else bool(sph.spher)
    use_gg = "p_GG_corr" in table.columns and not spher_ok
    return {"table": table, "F": float(eff["F"]), "df_effect": int(eff["DF"]), "df_error": int(table.iloc[1]["DF"]),
            "p": float(eff["p_GG_corr"] if use_gg else eff["p_unc"]), "p_uncorrected": float(eff["p_unc"]),
            "np2": float(eff["np2"]), "eps": float(eff["eps"]) if "eps" in table.columns else np.nan,
            "mauchly_W": None if sph is None else float(sph.W), "mauchly_p": np.nan if sph is None else float(sph.pval),
            "spher_ok": spher_ok, "correction": "Greenhouse-Geisser" if use_gg else "none"}

def posthoc_tables(wide, metric, alpha=0.05):
    """Tukey HSD (independent samples) next to Holm-corrected paired t-tests (Hedges g) for every pair of methods.

    Returns:
        (tukey_df, paired_df, merged_df) with ``disagree`` marking pairs where the two decisions differ.
        ``meandiff`` is mean(B) - mean(A) with A before B in column order.
    """
    long = _to_long(wide, metric)
    tk = pairwise_tukeyhsd(long[metric], long["method"], alpha=alpha)
    tk_df = pd.DataFrame(tk.summary().data[1:], columns=tk.summary().data[0])
    pw = pg.pairwise_tests(data=long, dv=metric, within="method", subject="fold_id", padjust="holm", effsize="hedges")
    rows = []
    for _, r in pw.iterrows():
        a, b = r["A"], r["B"]
        t = tk_df[((tk_df.group1 == a) & (tk_df.group2 == b)) | ((tk_df.group1 == b) & (tk_df.group2 == a))].iloc[0]
        rows.append({"A": a, "B": b, "meandiff": wide[b].mean() - wide[a].mean(), "tukey_p": float(t["p-adj"]),
                     "tukey_reject": bool(t["reject"]), "paired_t": float(r["T"]), "paired_p_holm": float(r["p_corr"]),
                     "paired_reject": bool(r["p_corr"] < alpha), "hedges_g": float(r["hedges"])})
    merged = pd.DataFrame(rows)
    merged["disagree"] = merged["tukey_reject"] != merged["paired_reject"]
    return tk_df, pw, merged

def nadeau_bengio(wide, n_test, n_train, alpha=0.05):
    """Nadeau-Bengio corrected resampled paired t-test for every pair of methods.

    The variance of the mean fold-level difference is inflated from ``var/J`` to ``(1/J + n_test/n_train) * var`` because the
    J resampled folds share training data and are positively correlated. ``A`` is the method with the higher mean,
    ``mean_diff = mean(A - B) >= 0``. p-values are Holm-adjusted over all pairs.
    """
    J = len(wide)
    order = wide.mean().sort_values(ascending=False).index.tolist()
    tcrit = sps.t.ppf(1 - alpha / 2, J - 1)
    rows = []
    for i, a in enumerate(order):
        for b in order[i + 1:]:
            d = wide[a] - wide[b]
            se = np.sqrt((1.0 / J + n_test / n_train) * d.var(ddof=1))
            se_naive = d.std(ddof=1) / np.sqrt(J)
            t = d.mean() / se if se > 0 else np.inf
            rows.append({"A": a, "B": b, "mean_diff": d.mean(), "se": se, "t": t, "p": 2 * sps.t.sf(abs(t), J - 1),
                         "ci_low": d.mean() - tcrit * se, "ci_high": d.mean() + tcrit * se,
                         "naive_ci_low": d.mean() - tcrit * se_naive, "naive_ci_high": d.mean() + tcrit * se_naive})
    out = pd.DataFrame(rows)
    out["p_holm"] = multipletests(out["p"], method="holm")[1]
    out["ci_excludes_zero"] = (out["ci_low"] > 0) | (out["ci_high"] < 0)
    return out

def nonparametric_tests(wide):
    """Friedman test followed by Holm-corrected Wilcoxon signed-rank tests on all pairs."""
    fr = sps.friedmanchisquare(*[wide[c] for c in wide]) if wide.shape[1] >= 3 else None
    rows = []
    cols = list(wide.columns)
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            d = wide[a] - wide[b]
            w = sps.wilcoxon(wide[a], wide[b]) if np.any(d != 0) else None
            rows.append({"A": a, "B": b, "median_diff": float(np.median(d)), "W": np.nan if w is None else float(w.statistic),
                         "p": 1.0 if w is None else float(w.pvalue)})
    wil = pd.DataFrame(rows)
    wil["p_holm"] = multipletests(wil["p"], method="holm")[1]
    wil["reject"] = wil["p_holm"] < 0.05
    return {"friedman_chi2": np.nan if fr is None else float(fr.statistic), "friedman_p": np.nan if fr is None else float(fr.pvalue),
            "wilcoxon": wil}

def lead_verdict(ci_excludes_zero, p_holm, wilcoxon_reject=None, badly_violated=False, alpha=0.05):
    """Is a lead defensible? 'defensible' = Nadeau-Bengio interval excludes zero AND the Holm-adjusted p < alpha (and, when the
    parametric assumptions are badly violated, the Wilcoxon-Holm test agrees); 'suggestive' = interval excludes zero but
    the lead does not survive the multiplicity correction; 'not distinguishable' = interval includes zero."""
    if not ci_excludes_zero:
        return "not distinguishable"
    ok = p_holm < alpha
    if badly_violated and wilcoxon_reject is not None:
        ok = ok and bool(wilcoxon_reject)
    return "defensible" if ok else "suggestive"

def _wilcoxon_pair(wil, x, y):
    """(reject, p_holm) of the Wilcoxon-Holm test for the pair {x, y}; (None, nan) if unavailable."""
    if wil is None:
        return None, np.nan
    r = wil[((wil.A == x) & (wil.B == y)) | ((wil.A == y) & (wil.B == x))]
    return (None, np.nan) if r.empty else (bool(r.iloc[0]["reject"]), float(r.iloc[0]["p_holm"]))

def final_summary(wide, nb, wilcoxon=None, badly_violated=False):
    """Methods ranked by mean score; last columns = Nadeau-Bengio 95 % interval of (method - best) and the lead verdict."""
    d = descriptive_table(wide)
    best = d.index[0]
    def vs_best(m):
        if m == best:
            return "best"
        r = nb[(nb.A == best) & (nb.B == m)].iloc[0]
        return f"{-r.mean_diff:+.3f} [{-r.ci_high:+.3f}, {-r.ci_low:+.3f}]"
    def verdict(m):
        if m == best:
            return "best"
        r = nb[(nb.A == best) & (nb.B == m)].iloc[0]
        return lead_verdict(bool(r.ci_excludes_zero), r.p_holm, _wilcoxon_pair(wilcoxon, best, m)[0], badly_violated)
    out = d[["mean", "sd", "median"]].copy()
    out["vs_best_95ci"] = [vs_best(m) for m in out.index]
    out["verdict"] = [verdict(m) for m in out.index]
    return out

In [ ]:
# TEST (written first): statistics on simulated fold-level scores with known structure
def _simulate_scores(means, n_folds=25, fold_sd=0.05, noise_sd=0.01, seed=0, task="dmax"):
    rng = np.random.default_rng(seed)
    names = [f"M{i + 1}" for i in range(len(means))]
    fold_eff = rng.normal(0, fold_sd, n_folds)
    M = np.asarray(means)[None, :] + fold_eff[:, None] + rng.normal(0, noise_sd, (n_folds, len(means)))
    rows = [{"task": task, "method": m, "repeat": f // 5, "fold": f % 5, "fold_id": f, "r2": M[f, j], "roc_auc": M[f, j],
             "n_train": 400, "n_test": 100, "status": "ok"} for f in range(n_folds) for j, m in enumerate(names)]
    return pd.DataFrame(rows)

def _test_stats():
    from scipy import stats as sps
    res = _simulate_scores([0.30, 0.31, 0.30, 0.40, 0.25])
    wide, dropped = wide_scores(res, "dmax", "r2")
    assert wide.shape == (25, 5) and dropped == 0 and list(wide.columns) == ["M1", "M2", "M3", "M4", "M5"]
    # complete cases: a failed cell removes that fold from every method
    res_bad = res.copy(); res_bad.loc[(res_bad.method == "M2") & (res_bad.fold_id == 3), ["r2", "status"]] = [np.nan, "error"]
    w2, d2 = wide_scores(res_bad, "dmax", "r2"); assert w2.shape == (24, 5) and d2 == 1 and 3 not in w2.index
    # descriptive
    desc = descriptive_table(wide)
    assert list(desc.index)[0] == "M4" and {"mean", "sd", "median", "iqr", "min", "max"} <= set(desc.columns)
    np.testing.assert_allclose(desc.loc["M4", "iqr"], np.subtract(*np.percentile(wide["M4"], [75, 25])))
    # assumptions
    a = assumption_checks(wide)
    assert a["levene_p"] > 0.01 and not a["badly_violated"] and "holds" in a["verdict"]
    rng = np.random.default_rng(1)
    het = wide.copy(); het["M5"] = het["M5"].mean() + 0.3 * rng.standard_cauchy(len(het)).clip(-5, 5)   # wild variance + outliers
    ah = assumption_checks(het); assert ah["levene_p"] < 0.01 and ah["badly_violated"] and "violated" in ah["verdict"]
    # repeated-measures ANOVA
    rm = rm_anova_report(wide, "r2")
    assert rm["p"] < 1e-10 and 0.5 < rm["np2"] <= 1 and rm["df_effect"] == 4 and rm["df_error"] == 96
    assert rm["correction"] in {"none", "Greenhouse-Geisser"} and 0 <= rm["mauchly_p"] <= 1
    null = rm_anova_report(wide_scores(_simulate_scores([0.3] * 5, seed=5), "dmax", "r2")[0], "r2")
    assert null["p"] > 0.01, null
    # post-hoc: Tukey ignores the pairing, the paired tests do not -> they disagree for the small consistent M1-M3 gap? no:
    # M2 - M1 = +0.01 with fold sd 0.05: invisible to Tukey, obvious to paired tests
    tk, pw, both = posthoc_tables(wide, "r2")
    row = both[(both.A == "M1") & (both.B == "M2")].iloc[0]
    assert (not row.tukey_reject) and row.paired_reject and row.disagree, row.to_dict()
    assert both.shape[0] == 10 and {"tukey_p", "paired_p_holm", "hedges_g", "meandiff"} <= set(both.columns)
    # Nadeau-Bengio corrected paired t-test
    nb = nadeau_bengio(wide, n_test=100, n_train=400)
    r = nb[(nb.A == "M4") & (nb.B == "M1")].iloc[0]
    d = wide["M4"] - wide["M1"]; J = len(d)
    se = np.sqrt((1 / J + 100 / 400) * d.var(ddof=1)); tcrit = sps.t.ppf(0.975, J - 1)
    np.testing.assert_allclose([r["mean_diff"], r["ci_low"], r["ci_high"]], [d.mean(), d.mean() - tcrit * se, d.mean() + tcrit * se])
    naive = tcrit * d.std(ddof=1) / np.sqrt(J)
    np.testing.assert_allclose((r["ci_high"] - r["ci_low"]) / 2 / naive, np.sqrt(1 + J * 100 / 400), rtol=1e-9)
    assert nb.shape[0] == 10 and {"p", "p_holm"} <= set(nb.columns)
    # a single wild method (one diverged fold) must flag the assumptions even if Levene stays quiet
    rng2 = np.random.default_rng(0)
    M7 = 0.3 + rng2.normal(0, 0.05, 25)[:, None] + rng2.normal(0, 0.04, (25, 7)); M7[0, 6] -= 0.4    # one diverged fold in one method
    ao = assumption_checks(pd.DataFrame(M7, columns=[f"M{i + 1}" for i in range(7)]))
    assert ao["levene_p"] > 0.01 and min(p for _, p in ao["shapiro"].values()) < 1e-3
    assert ao["badly_violated"], "a single method with Shapiro p < 0.001 must flag the normality assumption"
    # three-way verdict of a lead: defensible / suggestive / not distinguishable
    assert lead_verdict(False, 0.001) == "not distinguishable"
    assert lead_verdict(True, 0.01) == "defensible" and lead_verdict(True, 0.20) == "suggestive"
    assert lead_verdict(True, 0.01, wilcoxon_reject=False, badly_violated=True) == "suggestive"
    assert lead_verdict(True, 0.01, wilcoxon_reject=True, badly_violated=True) == "defensible"
    assert lead_verdict(True, 0.01, wilcoxon_reject=False, badly_violated=False) == "defensible"
    # non-parametric fallback
    npres = nonparametric_tests(wide)
    assert npres["friedman_p"] < 1e-10 and npres["wilcoxon"].shape[0] == 10 and "p_holm" in npres["wilcoxon"]
    # final ranking with the corrected interval against the best method
    summ = final_summary(wide, nb, npres["wilcoxon"], badly_violated=False)
    assert list(summ.index)[0] == "M4" and summ.iloc[0]["vs_best_95ci"] == "best" and summ.iloc[0]["verdict"] == "best"
    assert (summ["verdict"].iloc[1:] == "defensible").all()
    assert "[" in summ.loc["M1", "vs_best_95ci"] and summ["mean"].is_monotonic_decreasing
    print("statistics: OK")

_test_stats()

In [ ]:
METRIC_LABEL = {"r2": "R²", "roc_auc": "ROC-AUC"}

def m2_weight_table(res_task):
    """Per-expert stacking weight shares across folds (rows of M2): mean, sd, min, max and share of folds below 0.02."""
    rows = []
    for d in res_task[res_task["method"].str.startswith("M2") & (res_task["status"] == "ok")]["diag"]:
        dd = json.loads(d)
        if "weight_shares" in dd:
            rows.append(dict(zip(dd["experts"], dd["weight_shares"])))
    if not rows:
        return pd.DataFrame()
    w = pd.DataFrame(rows)
    return pd.DataFrame({"mean": w.mean(), "sd": w.std(ddof=1), "min": w.min(), "max": w.max(),
                         "folds_below_0.02": (w < 0.02).mean()})

def _col(wide, prefix):
    return next((c for c in wide.columns if c.startswith(prefix)), None)

def _pair(nb, x, y):
    """Mean(x - y) and its Nadeau-Bengio interval from the (A = higher-mean) table."""
    r = nb[((nb.A == x) & (nb.B == y)) | ((nb.A == y) & (nb.B == x))].iloc[0]
    s = 1.0 if r.A == x else -1.0
    lo, hi = sorted([s * r.ci_low, s * r.ci_high])
    return s * r.mean_diff, lo, hi, r.ci_excludes_zero, r.p_holm

def conclusion_paragraph(task, metric, wide, nb, summ, assum, rm, w2, npr):
    """Plain-language conclusion for one task, generated from the numbers: winner and whether its lead is defensible, the ridge
    baseline and the pure-fusion strategies, M4 vs M3 (do cross-block interactions carry signal?), the M2 weights."""
    lab = METRIC_LABEL[metric]
    bad = assum["badly_violated"]
    order = list(summ.index)
    best, second = order[0], order[1]
    wil = npr["wilcoxon"]
    n_pairs = len(nb)
    def verdict_of(x, y):
        """(diff x - y, lo, hi, verdict of 'x is better than y', NB Holm p, Wilcoxon-Holm p)"""
        d, lo, hi, excl, p = _pair(nb, x, y)
        wr, wp = _wilcoxon_pair(wil, x, y)
        return d, lo, hi, (lead_verdict(excl, p, wr, bad) if d > 0 else "not distinguishable"), p, wp
    d, lo, hi, v, p, wp = verdict_of(best, second)
    how = {"defensible": "so the lead is defensible",
           "suggestive": (f"so the lead is suggestive but not yet defensible: the interval excludes zero, but it does not survive "
                          f"Holm correction over {n_pairs} pairwise comparisons"),
           "not distinguishable": "so the lead is not defensible: the interval includes zero, i.e. it is within the noise of the resampling"}[v]
    txt = (f"**{task}** ({lab}, {len(wide)} complete folds): {best} ranks first with a mean of {wide[best].mean():.3f}. "
           f"Its lead over the runner-up {second} is {d:+.3f} (Nadeau-Bengio 95% interval [{lo:+.3f}, {hi:+.3f}], Holm-adjusted p = {p:.3g}; "
           f"Wilcoxon-Holm p = {wp:.3g}), {how}. ")
    others = order[1:]
    verd = {m: summ.loc[m, "verdict"] for m in others}
    indist = [m for m in others if verd[m] == "not distinguishable"]
    sugg = [m for m in others if verd[m] == "suggestive"]
    cred = [m for m in others if verd[m] == "defensible"]
    parts = []
    if cred:
        parts.append(f"credibly worse: {', '.join(cred)}")
    if sugg:
        parts.append(f"suggestively worse: {', '.join(sugg)}")
    if indist:
        parts.append(f"not distinguishable from the best: {', '.join(indist)}")
    txt += "Against the best method, " + "; ".join(parts) + ". "
    m1, m3, m4 = _col(wide, "M1"), _col(wide, "M3"), _col(wide, "M4")
    if m1:
        beat = [m for m in order if m != m1 and verdict_of(m, m1)[3] == "defensible"]
        fusion = [c for c in (_col(wide, "M2"), _col(wide, "M5"), _col(wide, "M6")) if c]
        fusion_beat = [m for m in fusion if m in beat]
        if m1 == best:
            txt += "The simple ridge baseline M1 is itself the best method, so none of the fusion strategies earned its complexity. "
        else:
            d1, l1, h1, v1, _, _ = verdict_of(best, m1)
            txt += (f"The ridge baseline {m1} trails the best by {d1:.3f} [{l1:+.3f}, {h1:+.3f}] ({v1}). "
                    + (f"Credibly better than M1: {', '.join(beat)}. " if beat else "Nothing credibly beats M1, so fusion complexity buys nothing at this sample size. ")
                    + (f"Of the dedicated fusion strategies ({', '.join(fusion)}), {', '.join(fusion_beat) if fusion_beat else 'none'} "
                       f"{'beat' if fusion_beat else 'beats'} the ridge baseline credibly. " if fusion else ""))
    if m3 and m4:
        dd, l4, h4, v4, p4, _ = verdict_of(m4, m3)
        verdict_txt = {"defensible": "carry real signal beyond the additive effects",
                       "suggestive": f"show suggestive but not established signal beyond the additive effects (Holm p = {p4:.3g})",
                       "not distinguishable": "do not add reliably to the additive GP: an informative negative result that makes M5/M6 unlikely to help"}[v4]
        txt += f"Cross-block interactions (M4 - M3 = {dd:+.3f} [{l4:+.3f}, {h4:+.3f}]) {verdict_txt}. "
    if len(w2):
        weak = w2[(w2["mean"] < 0.06) | (w2["folds_below_0.02"] >= 0.2)]
        if len(weak):
            txt += ("In the M2 stack " + "; ".join(f"the {e} expert has a mean weight share of {r['mean']:.3f} (below 0.02 in {100 * r['folds_below_0.02']:.0f}% of folds)"
                                                   for e, r in weak.iterrows())
                    + ": that encoder is redundant or uninformative and dropping it may help more than any fusion architecture. ")
        else:
            lowest = w2["mean"].idxmin()
            txt += f"All M2 experts keep a non-negligible weight across folds (smallest mean share: {lowest}, {w2.loc[lowest, 'mean']:.3f}). "
    if bad:
        txt += (f"The parametric assumptions are badly violated, so the primary tests are Friedman (chi2 = {npr['friedman_chi2']:.1f}, "
                f"p = {npr['friedman_p']:.3g}) followed by Wilcoxon-Holm; the t-based Nadeau-Bengio intervals are then only approximate "
                f"(RM-ANOVA for reference: F = {rm['F']:.2f}, p = {rm['p']:.3g}, partial eta² = {rm['np2']:.2f}).")
    else:
        txt += (f"The parametric assumptions are acceptable, so the parametric results are reported (RM-ANOVA F = {rm['F']:.2f}, "
                f"p = {rm['p']:.3g}, partial eta² = {rm['np2']:.2f}).")
    return txt

def analyze_task(res, task, out_dir=None, show=True, methods=None):
    """Steps 1-6 of the statistical comparison for one task; returns all intermediate tables and the conclusion."""
    metric, lab = PRIMARY_METRIC[task], METRIC_LABEL[PRIMARY_METRIC[task]]
    out_dir = Path(out_dir or RESULTS_DIR); out_dir.mkdir(parents=True, exist_ok=True)
    wide, dropped = wide_scores(res, task, metric, methods)
    res_t = res[(res["task"] == task)]
    print(f"\n{'=' * 100}\nTASK {task}  -  metric: {lab}  -  {len(wide)} complete folds x {wide.shape[1]} methods "
          f"({dropped} fold(s) dropped because a method failed)\n{'=' * 100}")
    out = {"metric": metric, "wide": wide}
    fin = (lambda name: (plt.savefig(out_dir / f"{task}_{name}.png", dpi=120), plt.show() if show else plt.close())[0])

    # ---- Step 1: descriptive
    desc = descriptive_table(wide); out["descriptive"] = desc
    print(f"\nStep 1 - descriptive statistics of {lab} over folds"); display(desc.round(4))
    too_few = len(wide) < wide.shape[1] + 2          # Mauchly's test and the repeated-measures ANOVA need n_folds > n_methods
    if too_few:
        out["skipped"] = True
    order = list(desc.index)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    long = _to_long(wide[order], lab)
    sns.boxplot(data=long, x="method", y=lab, order=order, color="0.9", fliersize=0, ax=axes[0])
    sns.stripplot(data=long, x="method", y=lab, order=order, color="C0", size=4, alpha=0.7, jitter=0.15, ax=axes[0])
    axes[0].set_title(f"{task}: {lab} per fold (box = IQR, points = individual folds)"); axes[0].tick_params(axis="x", rotation=30)
    for fid, row in wide[order].iterrows():
        axes[1].plot(range(len(order)), row.values, color="0.5", alpha=0.35, lw=0.8)
    axes[1].plot(range(len(order)), wide[order].mean().values, color="C3", lw=2.5, marker="o", label="mean over folds")
    axes[1].set_xticks(range(len(order))); axes[1].set_xticklabels(order, rotation=30)
    axes[1].set(title=f"{task}: paired across folds (one line per fold)", ylabel=lab); axes[1].legend()
    plt.tight_layout(); fin("descriptive")
    if too_few:
        print(f"\nOnly {len(wide)} complete folds for {wide.shape[1]} methods: too few for the repeated-measures tests "
              f"(need more folds than methods) -> statistics skipped (FAST_MODE is for debugging only).")
        return out

    # ---- Step 2: assumptions
    a = assumption_checks(wide); out["assumptions"] = a
    print("\nStep 2 - assumption checks"); print(a["verdict"])
    display(pd.DataFrame({"shapiro_W": {m: v[0] for m, v in a["shapiro"].items()}, "shapiro_p": {m: v[1] for m, v in a["shapiro"].items()}}).round(4))
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sps.probplot(a["residuals"].to_numpy().ravel(), dist="norm", plot=axes[0]); axes[0].set_title("QQ plot of RM-ANOVA residuals")
    sns.histplot(a["residuals"].to_numpy().ravel(), kde=True, ax=axes[1]); axes[1].set(title="residuals", xlabel="residual")
    plt.tight_layout(); fin("assumptions")

    # ---- Step 3: repeated-measures ANOVA
    rm = rm_anova_report(wide, metric); out["rm_anova"] = rm
    print("\nStep 3 - repeated-measures ANOVA (subjects = folds)"); display(rm["table"].round(5))
    print(f"Mauchly sphericity W = {rm['mauchly_W']}, p = {rm['mauchly_p']:.3g} -> sphericity "
          f"{'holds' if rm['spher_ok'] else 'is violated; Greenhouse-Geisser correction applied (epsilon = %.3f)' % rm['eps']}.")
    print(f"F({rm['df_effect']}, {rm['df_error']}) = {rm['F']:.3f}, p = {rm['p']:.3g} (correction: {rm['correction']}), "
          f"partial eta squared = {rm['np2']:.3f}")

    # ---- Step 4: post-hoc, Tukey vs paired
    tk, pw, both = posthoc_tables(wide, metric); out["posthoc"] = both
    print("\nStep 4 - post-hoc: Tukey HSD (independent samples) vs Holm-corrected paired t-tests (Hedges g)")
    show_cols = ["A", "B", "meandiff", "tukey_p", "tukey_reject", "paired_p_holm", "paired_reject", "hedges_g", "disagree"]
    display(both[show_cols].round(4))
    nd = int(both["disagree"].sum())
    print(f"{nd} of {len(both)} pairs get different decisions from the two procedures; where they disagree, trust the paired version "
          f"(Tukey ignores that every method is scored on the same folds).")

    # ---- Step 5: Nadeau-Bengio
    n_test, n_train = res_t["n_test"].mean(), res_t["n_train"].mean()
    nb = nadeau_bengio(wide, n_test, n_train); out["nadeau_bengio"] = nb
    print(f"\nStep 5 - Nadeau-Bengio corrected paired t-tests (variance inflation 1/J + n_test/n_train = "
          f"1/{len(wide)} + {n_test:.0f}/{n_train:.0f} = {1 / len(wide) + n_test / n_train:.3f} vs {1 / len(wide):.3f} uncorrected)")
    display(nb.round(4))
    best = order[0]
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(order) + 1.5))
    for i, m in enumerate(order[1:]):
        r = nb[(nb.A == best) & (nb.B == m)].iloc[0]
        ax.plot([-r.naive_ci_high, -r.naive_ci_low], [i + 0.12] * 2, color="C1", lw=3, label="naive paired CI" if i == 0 else None)
        ax.plot([-r.ci_high, -r.ci_low], [i - 0.12] * 2, color="k", lw=3, label="Nadeau-Bengio CI" if i == 0 else None)
        ax.plot(-r.mean_diff, i, "o", color="C3")
    ax.axvline(0, color="0.4", ls="--"); ax.set_yticks(range(len(order) - 1)); ax.set_yticklabels(order[1:])
    ax.set(xlabel=f"mean {lab} difference vs best ({best}); 95% intervals", title=f"{task}: each method minus the best"); ax.legend()
    plt.tight_layout(); fin("vs_best")

    # ---- Step 6: non-parametric fallback
    npr = nonparametric_tests(wide); out["nonparametric"] = npr
    print("\nStep 6 - non-parametric tests: Friedman " + f"chi2 = {npr['friedman_chi2']:.2f}, p = {npr['friedman_p']:.3g}; Wilcoxon signed-rank (Holm):")
    display(npr["wilcoxon"].round(4))
    print(("-> assumptions are badly violated: REPORT THE FRIEDMAN/WILCOXON RESULTS INSTEAD OF THE PARAMETRIC ONES."
           if a["badly_violated"] else "-> assumptions are acceptable; the non-parametric tests are shown as a robustness check."))

    # ---- summary, M2 weights, conclusion
    summ = final_summary(wide, nb, npr["wilcoxon"], a["badly_violated"]); out["summary"] = summ
    print(f"\nFinal summary, ranked by mean {lab} (last column: Nadeau-Bengio 95% interval of method - best)"); display(summ.round(4))
    w2 = m2_weight_table(res_t); out["m2_weights"] = w2
    if len(w2):
        print("\nM2 stacking weight shares across folds (a near-zero expert is redundant or uninformative):"); display(w2.round(3))
        fig, ax = plt.subplots(figsize=(6, 3.5))
        rows = [dict(zip(json.loads(d)["experts"], json.loads(d)["weight_shares"])) for d in
                res_t[res_t["method"].str.startswith("M2") & (res_t["status"] == "ok")]["diag"]]
        sns.boxplot(data=pd.DataFrame(rows), color="0.9", fliersize=0, ax=ax); sns.stripplot(data=pd.DataFrame(rows), color="C0", size=3, ax=ax)
        ax.set(title=f"{task}: M2 expert weight shares per fold", ylabel="share of stack weight")
        plt.tight_layout(); fin("m2_weights")
    out["best"] = best
    out["paragraph"] = conclusion_paragraph(task, metric, wide, nb, summ, a, rm, w2, npr)
    return out

In [ ]:
# TEST (written first): the per-task analysis runs end to end and produces the documented outputs
def _test_analyze():
    import shutil
    res = pd.concat([_simulate_scores([0.30, 0.31, 0.30, 0.40, 0.25], seed=0, task="dmax"),
                     _simulate_scores([0.30, 0.31, 0.30, 0.40, 0.25], seed=1, task="activity")], ignore_index=True)
    res["fit_time"] = 1.0
    res["diag"] = [json.dumps({"experts": ["fingerprint", "descriptors", "cell", "poi", "metadata"],
                               "weight_shares": [0.4, 0.001, 0.3, 0.299, 0.0]}) if m == "M2" else "{}" for m in res.method]
    out = RESULTS_DIR / "unittest_stats"; shutil.rmtree(out, ignore_errors=True); out.mkdir(parents=True)
    r = analyze_task(res, "dmax", out_dir=out, show=False)
    assert r["metric"] == "r2" and r["best"] == "M4" and r["wide"].shape == (25, 5)
    for k in ["descriptive", "assumptions", "rm_anova", "posthoc", "nadeau_bengio", "nonparametric", "summary", "paragraph"]:
        assert k in r, k
    par = r["paragraph"]
    assert "M4" in par and "Nadeau" in par and "Wilcoxon" in par and "defensible" in par
    assert "descriptors" in par and "metadata" in par, "weak M2 experts must be named"
    assert "M3" in par and "interaction" in par
    # the metric follows the task: activity is analysed on ROC-AUC
    assert analyze_task(res, "activity", out_dir=out, show=False)["metric"] == "roc_auc"
    # the M2 weight check names an expert that is (almost) never used
    w = m2_weight_table(res[res.task == "dmax"])
    assert w.loc["descriptors", "folds_below_0.02"] == 1.0 and w.loc["fingerprint", "mean"] > 0.3
    assert len(list(out.glob("*.png"))) >= 3
    # FAST_MODE-sized data (3 folds, 5 methods): no valid repeated-measures test -> descriptive output only, no exception
    few = res[(res.task == "dmax") & (res.fold_id < 3)]
    r3 = analyze_task(few, "dmax", out_dir=out, show=False)
    assert r3.get("skipped") and "descriptive" in r3 and "rm_anova" not in r3 and r3["wide"].shape == (3, 5)
    shutil.rmtree(out, ignore_errors=True)
    print("analyze_task: OK")

_test_analyze()

### 8.1 Per-task analysis
For each task: (1) descriptive statistics, box plot with fold points and the paired-across-folds plot, (2) Levene and
Shapiro-Wilk checks with a QQ plot, (3) repeated-measures ANOVA with Mauchly sphericity / Greenhouse-Geisser and partial
eta², (4) Tukey HSD **next to** Holm-corrected paired tests (trust the paired ones), (5) Nadeau-Bengio corrected
intervals, (6) Friedman + Wilcoxon-Holm, followed by the ranked summary table and the M2 weight check.

In [ ]:
STATS = {}
for t in TASKS:
    if t not in set(results["task"]):
        continue
    STATS[t] = analyze_task(results, t)

### Why the Nadeau-Bengio intervals are wider than Tukey's, and which one to report
* **Tukey HSD** treats the 25 fold scores of every method as 25 independent observations. That is wrong twice over: the
  *same* folds are shared by all methods (the scores are paired, so fold difficulty should be removed, which makes
  differences *easier* to detect), and the folds overlap in their training data (the scores of different folds are
  positively correlated, which makes the nominal sample size of 25 an over-statement).
* **Paired tests** (Holm-corrected `pairwise_tests`) fix the first problem but still assume 25 independent differences, so
  their intervals are too narrow.
* **Nadeau-Bengio** fixes the second: the variance of the mean difference is `(1/J + n_test/n_train) * s²` instead of
  `s²/J`. With 5-fold CV `n_test/n_train = 1/4`, so the variance is about 7x larger and the intervals about 2.7x wider than
  the naive paired ones. This is the honest width of what a resampling experiment can say about generalisation.
* **Report** the Nadeau-Bengio corrected intervals (and the Holm-adjusted paired tests as the significance statement) in a
  paper; the Tukey table is shown only to illustrate how much the independence assumption distorts the picture. If the
  assumption checks fail badly, report Friedman + Wilcoxon-Holm instead; note that rank-based tests share the
  fold-overlap problem, so their p-values are also optimistic.

### 8.2 Parity plots of the best fold of every method
Measured (x) against predicted (y) target values, **read from the per-fold predictions that the harness saved**
(`predictions/`), so nothing is retrained. For each task there is one panel per method, showing the fold (repeat, fold)
in which that method reaches its **highest primary metric** (R², ROC-AUC for activity) among the folds that have saved
predictions. The best fold differs between methods and is the most favourable of 25, so these plots show *what a good fit
looks like*, not the expected performance (the tables above are the honest comparison). The axes are shared across the
panels of a task; the dashed line is the ideal `y = x`. Activity has 0/1 labels: the measured class is jittered
horizontally and the red ticks are the mean predicted probability of each class.

In [ ]:
PARITY_LABEL = {"dmax": "Dmax (fraction)", "pdc50": "pDC50", "activity": "activity class (0 = inactive, 1 = active)"}

def best_fold_table(res, task, preds):
    """The fold with the highest primary metric of every method, among the folds that have saved predictions.

    Args:
        res: the results table (``results.csv``).
        task: task name.
        preds: dict ``(task, method, repeat, fold) -> {'y_true', 'y_pred', ...}`` as returned by ``load_predictions``.

    Returns:
        DataFrame with columns ``method, repeat, fold, score`` (one row per method, sorted by method name).
    """
    metric = PRIMARY_METRIC[task]
    r = res[(res["task"] == task) & (res["status"] == "ok")].dropna(subset=[metric])
    r = r[[(task, m, int(a), int(b)) in preds for m, a, b in zip(r["method"], r["repeat"], r["fold"])]]
    best = r.loc[r.groupby("method")[metric].idxmax()]
    return (best.rename(columns={metric: "score"})[["method", "repeat", "fold", "score"]]
            .astype({"repeat": int, "fold": int}).sort_values("method").reset_index(drop=True))

def parity_limits(arrays):
    """Shared ``(lo, hi)`` of the measured and predicted values of several panels, padded by 3 %."""
    allv = np.concatenate([np.asarray(a, float).ravel() for a in arrays])
    lo, hi = np.nanmin(allv), np.nanmax(allv)
    return lo - 0.03 * (hi - lo), hi + 0.03 * (hi - lo)

def parity_panel(ax, task, y_true, y_pred, lims=None, rng=None, flag=None, rmse=True):
    """Draw one measured-vs-predicted panel on ``ax``: scatter with the ideal ``y = x`` line and the RMSE; ``activity`` is the
    jittered measured class against the predicted probability with the class means as red ticks.

    Args:
        ax: matplotlib axes.
        task: task name.
        y_true, y_pred: measured and predicted values.
        lims: shared ``(lo, hi)`` axis range for the regression tasks (default: from these values).
        rng: ``numpy`` generator for the activity jitter.
        flag: optional boolean array; the ``True`` points are drawn in a second colour (``C1``) after the others.
        rmse: annotate the RMSE over all points of a regression panel (switch off when the title already reports the scores).
    """
    yt, yp = np.asarray(y_true, float), np.asarray(y_pred, float)
    rng = rng or np.random.default_rng(0)
    flag = np.zeros(len(yt), bool) if flag is None else np.asarray(flag, bool)
    xs = yt + rng.uniform(-0.12, 0.12, len(yt)) if task == "activity" else yt
    for sel, color in ((~flag, "C0"), (flag, "C1")):
        if sel.any():
            ax.scatter(xs[sel], yp[sel], s=8, alpha=0.35, color=color, edgecolors="none")
    if task == "activity":
        ax.plot([0, 1], [0, 1], "k--", lw=1)
        for c in (0, 1):
            if (yt == c).any():
                ax.hlines(yp[yt == c].mean(), c - 0.3, c + 0.3, color="C3", lw=2.5)
        ax.set(xlim=(-0.5, 1.5), ylim=(-0.03, 1.03), xticks=[0, 1], xticklabels=["inactive", "active"],
               xlabel="measured class", ylabel="predicted probability of activity")
    else:
        lo, hi = lims if lims is not None else parity_limits([yt, yp])
        ax.plot([lo, hi], [lo, hi], "k--", lw=1)
        ax.set(xlim=(lo, hi), ylim=(lo, hi), xlabel=f"measured {PARITY_LABEL[task]}", ylabel=f"predicted {PARITY_LABEL[task]}")
        ax.set_aspect("equal")
        if rmse:
            ax.text(0.04, 0.96, f"RMSE = {np.sqrt(np.mean((yt - yp) ** 2)):.3f}", transform=ax.transAxes, va="top", fontsize=9)

def parity_plots(res, preds, task, out_dir=None, show=True, ncols=4):
    """Measured (x) versus predicted (y) values of the best fold of every method, one panel per method (no refitting).

    Args:
        res: the results table (``results.csv``).
        preds: dict of saved per-fold predictions (``load_predictions``).
        task: task name; ``activity`` is drawn as the jittered measured class against the predicted probability.
        out_dir: where ``<task>_parity.png`` is saved (default ``RESULTS_DIR``).
        show: display the figure (otherwise close it).
        ncols: panels per row.

    Returns:
        (best-fold table, figure).
    """
    metric, lab = PRIMARY_METRIC[task], METRIC_LABEL[PRIMARY_METRIC[task]]
    out_dir = Path(out_dir or RESULTS_DIR); out_dir.mkdir(parents=True, exist_ok=True)
    best = best_fold_table(res, task, preds)
    data = [preds[(task, r.method, r.repeat, r.fold)] for r in best.itertuples()]
    n = len(best); nrows = max(1, int(np.ceil(n / ncols)))
    fig, axes = plt.subplots(nrows, max(1, min(ncols, n)) if n else ncols, figsize=(4.2 * ncols, 4.2 * nrows), squeeze=False)
    lims = parity_limits([np.concatenate([d["y_true"], d["y_pred"]]) for d in data]) if n else (0.0, 1.0)
    rng = np.random.default_rng(0)
    for ax, row, d in zip(axes.ravel(), best.itertuples(), data):
        parity_panel(ax, task, d["y_true"], d["y_pred"], lims=lims, rng=rng)
        yt = np.asarray(d["y_true"], float)
        ax.set_title(f"{row.method}  (repeat {row.repeat}, fold {row.fold})\n{lab} = {row.score:.3f}, n = {len(yt)}", fontsize=9)
    for ax in axes.ravel()[n:]:
        ax.set_visible(False)
    fig.suptitle(f"{task}: measured vs predicted, best fold of each method (highest {lab}; read from the saved fold predictions)", y=1.0)
    plt.tight_layout(); plt.savefig(out_dir / f"{task}_parity.png", dpi=120, bbox_inches="tight")
    plt.show() if show else plt.close(fig)
    return best, fig


In [ ]:
# TEST (written first): parity plots show the saved predictions of every method's best fold
def _test_parity():
    import shutil
    from sklearn.metrics import r2_score as _r2, roc_auc_score as _auc
    rng = np.random.default_rng(0)
    rows, preds = [], {}
    for task, metric in (("dmax", "r2"), ("activity", "roc_auc")):
        for j, m in enumerate(["M1", "M2", "M3", "M4"]):
            for f in range(4):
                n = 60
                noise = 0.02 if f == 2 else 0.15 + 0.05 * j                  # fold 2 is the best fold of every method
                if task == "dmax":
                    yt = rng.uniform(0, 1, n); yp = yt + rng.normal(0, noise, n); score = _r2(yt, yp)
                else:
                    yt = rng.integers(0, 2, n).astype(float); yp = np.clip(0.5 + (yt - 0.5) * (1 - 6 * noise) + rng.normal(0, 0.1, n), 0, 1)
                    score = _auc(yt, yp)
                row = {"task": task, "method": m, "repeat": 0, "fold": f, "status": "ok", "r2": np.nan, "roc_auc": np.nan}
                row[metric] = score
                if m != "M4":                                                 # M4 has results rows but no saved predictions: omitted
                    preds[(task, m, 0, f)] = {"test_idx": np.arange(n), "y_true": yt, "y_pred": yp}
                rows.append(row)
        # a fold with a tempting score but no saved predictions, and a failed fold, must both be ignored
        rows.append({"task": task, "method": "M1", "repeat": 1, "fold": 0, "status": "ok", "r2": 0.99, "roc_auc": 0.99})
        rows.append({"task": task, "method": "M2", "repeat": 1, "fold": 1, "status": "error", "r2": np.nan, "roc_auc": np.nan})
    res = pd.DataFrame(rows)
    out = RESULTS_DIR / "unittest_parity"; shutil.rmtree(out, ignore_errors=True)
    best = best_fold_table(res, "dmax", preds)
    assert list(best["method"]) == ["M1", "M2", "M3"] and (best["fold"] == 2).all() and (best["repeat"] == 0).all(), best
    table, fig = parity_plots(res, preds, "dmax", out_dir=out, show=False, ncols=2)
    visible = [a for a in fig.axes if a.get_visible()]
    assert len(fig.axes) == 4 and len(visible) == 3, "3 methods on a 2 x 2 grid: one hidden panel"
    for ax, m in zip(visible, ["M1", "M2", "M3"]):
        d = preds[("dmax", m, 0, 2)]
        pts = ax.collections[0].get_offsets()
        np.testing.assert_allclose(pts[:, 0], d["y_true"]); np.testing.assert_allclose(pts[:, 1], d["y_pred"])   # measured on x, predicted on y
        assert m in ax.get_title() and "fold 2" in ax.get_title()
        assert ax.get_xlim() == visible[0].get_xlim() and ax.get_ylim() == visible[0].get_ylim() and ax.get_xlim() == ax.get_ylim()
        assert "measured" in ax.get_xlabel().lower() or "measured" in visible[0].get_xlabel().lower()
    np.testing.assert_allclose(table["score"].to_numpy(), [_r2(preds[("dmax", m, 0, 2)]["y_true"], preds[("dmax", m, 0, 2)]["y_pred"]) for m in ["M1", "M2", "M3"]])
    assert (out / "dmax_parity.png").exists()
    # activity: predicted probability against the measured class (jittered), same fold-selection rule on ROC-AUC
    table_a, fig_a = parity_plots(res, preds, "activity", out_dir=out, show=False, ncols=3)
    assert (table_a["fold"] == 2).all() and (out / "activity_parity.png").exists()
    ax = [a for a in fig_a.axes if a.get_visible()][0]
    d = preds[("activity", "M1", 0, 2)]; pts = ax.collections[0].get_offsets()
    np.testing.assert_allclose(pts[:, 1], d["y_pred"]); assert (np.abs(pts[:, 0] - d["y_true"]) < 0.5).all()
    # ``flag`` draws a subset (e.g. molecules also in the training data) in a second colour, with the same coordinates
    flag = np.zeros(len(d["y_true"]), bool); flag[::3] = True
    for tk in ("dmax", "activity"):
        yt_, yp_ = (preds[("dmax", "M1", 0, 2)]["y_true"], preds[("dmax", "M1", 0, 2)]["y_pred"]) if tk == "dmax" else (d["y_true"], d["y_pred"])
        scatters = lambda a: [c for c in a.collections if type(c).__name__ == "PathCollection"]    # activity also has the class-mean ticks
        fg, ax = plt.subplots(); parity_panel(ax, tk, yt_, yp_, flag=flag)
        sc = scatters(ax)
        assert len(sc) == 2 and sc[0].get_offsets().shape[0] == (~flag).sum() and sc[1].get_offsets().shape[0] == flag.sum()
        np.testing.assert_allclose(sc[1].get_offsets()[:, 1], yp_[flag])
        fg2, ax2 = plt.subplots(); parity_panel(ax2, tk, yt_, yp_, flag=np.zeros(len(yt_), bool))
        assert len(scatters(ax2)) == 1, "an all-False flag draws one group only"
    # the in-panel RMSE (over all points) is on by default and can be switched off when the title already carries the scores
    yt_, yp_ = preds[("dmax", "M1", 0, 2)]["y_true"], preds[("dmax", "M1", 0, 2)]["y_pred"]
    fg, ax = plt.subplots(); parity_panel(ax, "dmax", yt_, yp_)
    assert len(ax.texts) == 1 and "RMSE" in ax.texts[0].get_text()
    fg2, ax2 = plt.subplots(); parity_panel(ax2, "dmax", yt_, yp_, rmse=False)
    assert len(ax2.texts) == 0
    plt.close("all"); shutil.rmtree(out, ignore_errors=True)
    print("parity plots: OK")

_test_parity()

In [ ]:
PREDS = load_predictions(RESULTS_DIR)                 # per-fold predictions saved by the harness: no model is refitted here
PARITY = {}
for t in TASKS:
    if t in set(results["task"]):
        PARITY[t], _ = parity_plots(results, PREDS, t)
display(pd.concat(PARITY, names=["task"]).round(3).rename_axis(index=["task", None]) if PARITY else "no results")

### 8.3 Parity plots on `PROTAC-test.csv`: the best-fold models, no retraining
The external table `data/yaochen/PROTAC-test.csv` is scored by the **fitted fold models of the best fold** of each method, without
refitting anything. This notebook does not store fitted models (only their predictions), so the models come from the exported members
of `ensemble_speed_rl.ipynb` (`ensemble_speed_results/members/`): member *k* of a task is the M4 / M7 model of cross-validation split
*k* (repeat x 5 + fold), fitted with the same rows and seed, and the cell checks that each member reproduces the predictions saved
here before using it. **Only M4 and M7 have stored models**; M1, M2, M3, M5 and M6 would have to be refitted and are therefore not
plotted (the cell lists them).

The features of the new rows (Morgan bits, RDKit descriptors, ESM-2 protein embeddings, cell-line embeddings) are computed once by the
cached loader (`TACKAI_CACHE`) and reused afterwards; every row is scored in its own context. Molecules are matched to the development
data by *canonical* SMILES: rows whose compound the models were trained on are drawn in **orange** (not out-of-sample, often the very same
record) and reported separately from the **unseen-compound** rows (blue). Censoring operators in the raw table are ignored as everywhere
in this notebook, so `DC50 > 25000 nM` entries enter as pDC50 = 4.6 (the column at the left edge of the pDC50 panels).

In [ ]:
import nbformat
from scipy.stats import spearmanr
from sklearn.metrics import r2_score, roc_auc_score

def compound_overlap(ext_tab, dev_tab):
    """Which rows of an external table repeat the development data (compared by canonical SMILES, so respelled molecules match).

    Args:
        ext_tab, dev_tab: tables with the columns ``smiles``, ``poi_uniprot`` and ``cell_key`` (``build_dev_table`` output).

    Returns:
        (seen, same_record): boolean arrays over the rows of ``ext_tab``; ``seen`` = the compound occurs in the development data,
        ``same_record`` = the compound occurs there with the same protein and cell line (the very same measurement).
    """
    canon = lambda s: (lambda m: None if m is None else Chem.MolToSmiles(m))(Chem.MolFromSmiles(s))
    dev_c = [canon(s) for s in dev_tab["smiles"]]
    ext_c = [canon(s) for s in ext_tab["smiles"]]
    dev_set = {c for c in dev_c if c is not None}
    dev_rec = {(c, str(p), str(k)) for c, p, k in zip(dev_c, dev_tab["poi_uniprot"], dev_tab["cell_key"]) if c is not None}
    seen = np.array([c is not None and c in dev_set for c in ext_c])
    same = np.array([c is not None and (c, str(p), str(k)) in dev_rec for c, p, k in zip(ext_c, ext_tab["poi_uniprot"], ext_tab["cell_key"])])
    return seen, same

def external_scores(task, y_true, y_pred):
    """Scores of an external prediction set: R2, RMSE, Spearman (ROC-AUC and Brier for ``activity``); NaN if fewer than 3 rows
    or, for activity, a single class."""
    yt, yp = np.asarray(y_true, float), np.asarray(y_pred, float)
    names = ["ROC-AUC", "Brier"] if task == "activity" else ["R2", "RMSE", "Spearman"]
    if len(yt) < 3 or (task == "activity" and len(np.unique(yt)) < 2):
        return dict.fromkeys(names, np.nan)
    if task == "activity":
        return {"ROC-AUC": roc_auc_score(yt, yp), "Brier": float(np.mean((yt - yp) ** 2))}
    return {"R2": r2_score(yt, yp), "RMSE": float(np.sqrt(np.mean((yt - yp) ** 2))), "Spearman": spearmanr(yt, yp)[0]}

def load_speed_cells(path, tags):
    """Execute the definition cells (by tag) of the speed notebook in this namespace: the plain-array engine that scores the
    exported fold models (``FoldEnsemble``, ``load_members``, ``predict_with_row_contexts``, ``predict_external``)."""
    snb = nbformat.read(path, as_version=4)
    by_tag = {c.metadata.get("tags", [""])[0]: c for c in snb.cells if c.cell_type == "code"}
    for t in tags:
        exec(compile(by_tag[t].source, f"<{Path(path).name}:{t}>", "exec"), globals())


In [ ]:
# TEST (written first): overlap of an external table with the development data, and its scores
def _test_external():
    dev = pd.DataFrame({"smiles": ["CCO", "c1ccccc1O", "CCN"], "poi_uniprot": ["P1", "P2", "P3"], "cell_key": ["C1", "C2", "C3"]})
    ext = pd.DataFrame({"smiles": ["OCC", "Oc1ccccc1", "CCCC", "not a smiles", "CCN"], "poi_uniprot": ["P1", "PX", "P1", "P1", "P3"],
                        "cell_key": ["C1", "C2", "C1", "C1", "CX"]})
    seen, same = compound_overlap(ext, dev)
    assert seen.tolist() == [True, True, False, False, True], "re-spelled molecules match; an invalid SMILES never matches"
    assert same.tolist() == [True, False, False, False, False], "same record = same compound, protein and cell line"
    rng = np.random.default_rng(0)
    yt = rng.uniform(0, 1, 50); yp = yt + rng.normal(0, 0.1, 50)
    s = external_scores("dmax", yt, yp)
    assert set(s) == {"R2", "RMSE", "Spearman"} and 0.5 < s["R2"] < 1 and abs(s["RMSE"] - np.sqrt(np.mean((yt - yp) ** 2))) < 1e-12
    lab = (rng.uniform(size=50) > 0.5).astype(float); a = external_scores("activity", lab, np.clip(lab * 0.6 + 0.2 + rng.normal(0, 0.1, 50), 0, 1))
    assert set(a) == {"ROC-AUC", "Brier"} and a["ROC-AUC"] > 0.9
    assert np.isnan(external_scores("dmax", yt[:2], yp[:2])["R2"]), "fewer than 3 rows: NaN, not an exception"
    assert np.isnan(external_scores("activity", np.ones(10), np.linspace(0, 1, 10))["ROC-AUC"]), "a single class has no ROC-AUC"
    print("external table helpers: OK")

_test_external()

In [ ]:
SPEED_NB = RESULTS_DIR.parent / "ensemble_speed_rl.ipynb"
MEMBERS_DIR = RESULTS_DIR.parent / "ensemble_speed_results" / "members"
TEST_FILE = REPO_ROOT / "data/yaochen/PROTAC-test.csv"
STORED = {"M4_gp_inter": "m4", "M7_xgb": "m7"}         # methods whose fitted fold models exist (exported members of the speed notebook)
missing = [str(p) for p in (SPEED_NB, MEMBERS_DIR, TEST_FILE) if not p.exists()]
if missing:
    print("skipped: not found:", ", ".join(missing), "(the fitted models are not stored by this notebook; run ensemble_speed_rl.ipynb first)")
else:
    load_speed_cells(SPEED_NB, ["engine", "train_members", "ext"])
    _saved_dev_files = DEV_FILES
    DEV_FILES = [TEST_FILE]
    try:
        ext = load_tack_dev_data()             # features of the new rows; ESM / cell embeddings computed once and cached in TACKAI_CACHE
        ext_tab = build_dev_table()
    finally:
        DEV_FILES = _saved_dev_files
    seen, same_record = compound_overlap(ext_tab, build_dev_table())
    print(f"{TEST_FILE.name}: {len(ext_tab)} rows, {ext_tab['smiles'].nunique()} unique SMILES, {ext_tab['poi_uniprot'].nunique()} proteins, "
          f"{ext_tab['cell_key'].nunique()} cell lines\n  compound also in the development data: {seen.sum()} rows (orange; same compound + protein + cell line: "
          f"{same_record.sum()} rows)\n  compound NOT in the development data: {(~seen).sum()} rows (blue, the out-of-sample subset)")
    X_ext = assemble_X(ext).astype(np.float64)  # also redefines BLOCKS (same layout as the development data)
    tasks_ext = [t for t in TASKS if t in set(results["task"]) and (MEMBERS_DIR / f"{t}__00.pkl").exists()]
    ext_pred, metric_rows, row_frames, picked = {}, [], [], {}
    for t in tasks_ext:
        bf = best_fold_table(results, t, PREDS)
        bf = bf[bf["method"].isin(STORED)].reset_index(drop=True)
        picked[t] = bf
        for r in bf.itertuples():
            member = load_members(MEMBERS_DIR, t, [r.repeat * N_SPLITS + r.fold])[0]
            saved = PREDS[(t, r.method, r.repeat, r.fold)]
            # the member IS that cross-validation fold's model: same test rows and the same predictions as saved by the harness
            assert member["repeat"] == r.repeat and member["fold"] == r.fold
            np.testing.assert_array_equal(member["test_idx"], saved["test_idx"])
            np.testing.assert_allclose(member[f"test_pred_{STORED[r.method]}"], saved["y_pred"], atol=5e-3)
            p = predict_external([member], X_ext, ext["y"][t])           # one member: no ensemble, exactly that model
            pred, sn = p[STORED[r.method]], seen[p["idx"]]
            ext_pred[(t, r.method)] = {"y_true": p["y_true"], "y_pred": pred, "seen": sn}
            for subset, sel in (("unseen compounds", ~sn), ("all rows", np.ones(len(sn), bool))):
                metric_rows.append({"task": t, "method": r.method, "best fold": f"repeat {r.repeat}, fold {r.fold}", "n": int(sel.sum()),
                                    **external_scores(t, p["y_true"][sel], pred[sel]), "subset": subset})
            row_frames.append(ext_tab.iloc[p["idx"]][["smiles", "poi_uniprot", "cell_key", "e3_raw", "assay_raw"]].assign(
                task=t, method=r.method, measured=p["y_true"], predicted=pred, compound_in_dev=sn, same_record_in_dev=same_record[p["idx"]]))
    print("no stored fitted model (refitting would be needed), not plotted: "
          + ", ".join(m for m in sorted(set(results["method"])) if m not in STORED))
    ext_metrics = pd.DataFrame(metric_rows)
    ext_metrics.to_csv(RESULTS_DIR / "protac_test_best_fold_metrics.csv", index=False)
    pd.concat(row_frames).to_csv(RESULTS_DIR / "protac_test_best_fold_predictions.csv", index=False)
    display(ext_metrics.round(3))
    methods_ext = list(STORED)
    fig, axes = plt.subplots(len(tasks_ext), len(methods_ext), figsize=(4.8 * len(methods_ext), 4.9 * len(tasks_ext)), squeeze=False)
    rng = np.random.default_rng(0)
    for ri, t in enumerate(tasks_ext):
        lims = parity_limits([a for m in methods_ext for a in (ext_pred[(t, m)]["y_true"], ext_pred[(t, m)]["y_pred"])])
        lab, col = ("ROC-AUC", "ROC-AUC") if t == "activity" else ("R²", "R2")
        for ci, m in enumerate(methods_ext):
            d = ext_pred[(t, m)]; ax = axes[ri, ci]
            parity_panel(ax, t, d["y_true"], d["y_pred"], lims=lims, rng=rng, flag=d["seen"], rmse=False)
            sc = {s: ext_metrics[(ext_metrics.task == t) & (ext_metrics.method == m) & (ext_metrics.subset == s)].iloc[0] for s in ("unseen compounds", "all rows")}
            bfr = picked[t][picked[t].method == m].iloc[0]
            ax.set_title(f"{t}: {m}, best fold (repeat {bfr['repeat']}, fold {bfr['fold']}; CV {lab} = {bfr['score']:.3f})\n"
                         f"unseen compounds (blue): {lab} = {sc['unseen compounds'][col]:.3f}, n = {int(sc['unseen compounds']['n'])}\n"
                         f"all rows (blue + orange): {lab} = {sc['all rows'][col]:.3f}, n = {int(sc['all rows']['n'])}", fontsize=8.5)
    fig.suptitle(f"{TEST_FILE.name}: measured vs predicted by the best-fold models (stored fold models, no retraining)\n"
                 "orange = compound also in the development data (training leak, not out-of-sample)", y=1.0)
    assert all("bound method" not in ax.get_title() for ax in fig.axes), "a title picked up a pandas method (row.repeat is Series.repeat)"
    plt.tight_layout(); plt.savefig(RESULTS_DIR / "parity_protac_test.png", dpi=120, bbox_inches="tight"); plt.show()

## 9. Conclusions
Generated from the numbers above (nothing is hand-written): the winner per task, whether its lead is defensible under the
Nadeau-Bengio interval, whether the simple ridge baseline is within noise, whether cross-block interactions carry signal
(M4 vs M3), and which M2 expert gets (almost) no weight.

In [ ]:
STATS = {t: s for t, s in STATS.items() if not s.get("skipped")}
if not STATS:
    print("No task has enough complete folds for the statistical tests (FAST_MODE?).")
summary_rows = []
for t, s in STATS.items():
    best = s["best"]; sm = s["summary"]
    nxt = sm.index[1]
    summary_rows.append({"task": t, "metric": METRIC_LABEL[s["metric"]], "winner": best, "mean": sm.loc[best, "mean"],
                         "runner_up": nxt, "gap_vs_runner_up": sm.loc[nxt, "vs_best_95ci"], "lead_verdict": sm.loc[nxt, "verdict"],
                         "not_distinguishable": int((sm["verdict"] == "not distinguishable").sum()),
                         "suggestively_worse": int((sm["verdict"] == "suggestive").sum()),
                         "credibly_worse": int((sm["verdict"] == "defensible").sum()),
                         "assumptions_badly_violated": s["assumptions"]["badly_violated"]})
if summary_rows:
    overview = pd.DataFrame(summary_rows).set_index("task")
    display(overview.round(4))
for t, s in STATS.items():
    display(Markdown(s["paragraph"]))